### Import library

In [ ]:
import pandas as pd
import numpy as np
import random
import datetime
from collections import defaultdict
import copy
import matplotlib.pyplot as plt
import mlflow
import os

print("Libraries imported successfully!")

### Dataset Course, Rooms, Slots - Setup

In [8]:
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
import mlflow

def configure_courses(csv_path):
    df_courses = pd.read_csv(csv_path)
    df_courses['event_id'] = [f"EVT_{i:04d}" for i in range(len(df_courses))]
    df_courses['requires_practicum_room'] = df_courses['course_name'].str.contains('Praktikum', case=False, na=False)
    df_courses['slots_needed'] = df_courses['course_time_minute'] // 50
    
    cols_order = ['event_id', 'course_id', 'course_name', 'requires_practicum_room', 
                  'slots_needed', 'teacher_id', 'teacher_priority']
    available_cols = [c for c in cols_order if c in df_courses.columns]
    return df_courses[available_cols]

def configure_rooms_and_timeslots(rooms_csv_path):
    df_rooms = pd.read_csv(rooms_csv_path)
    
    theory_rooms = {'F2.1', 'F2.2', 'F2.4', 'F2.5', 'F2.6', 'F2.8', 'F2.9', 'F3.1', 'F3.10',
                    'F3.11', 'F3.12', 'F3.13', 'F3.14', 'F3.15', 'F3.16', 'F3.17', 'F3.18',
                    'F3.2', 'F3.3', 'F3.4', 'F3.5a', 'F3.5b', 'F3.6', 'F3.7a', 'F3.7b',
                    'F3.8', 'F3.9', 'F4.10', 'F4.11', 'F4.12', 'F4.13', 'F4.14', 'F4.2',
                    'F4.3', 'F4.4', 'F4.5', 'F4.6', 'F4.7', 'F4.9'}
    
    practicum_rooms = {'G1.2', 'G1.3', 'G1.4', 'G1.5', 'G1.6', 
                       'F2.1', 'F2.2', 'F2.4', 'F2.5', 'F2.6', 'F2.8', 'F2.9'}
    
    df_rooms['is_theory'] = df_rooms['room_id'].isin(theory_rooms)
    df_rooms['is_practicum'] = df_rooms['room_id'].isin(practicum_rooms)

    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    granularity_minutes = 50
    all_timeslots = []
    slot_id_counter = 0
    
    for day in days:
        sessions = [
            {'type': 'Morning', 'start': "07:00", 'end': "12:00"},
            {'type': 'Afternoon', 'start': "12:30", 'end': "18:20"}
        ]
        for session in sessions:
            curr_time = datetime.strptime(session['start'], "%H:%M")
            end_time = datetime.strptime(session['end'], "%H:%M")
            while curr_time < end_time:
                all_timeslots.append({
                    'slot_id': f"T_{slot_id_counter:04d}",
                    'day': day,
                    'session_type': session['type'],
                    'time_start': curr_time.strftime("%H:%M"),
                    'time_end': (curr_time + timedelta(minutes=granularity_minutes)).strftime("%H:%M")
                })
                curr_time += timedelta(minutes=granularity_minutes)
                slot_id_counter += 1
                
    return df_rooms, pd.DataFrame(all_timeslots)

# Eksekusi Pembuatan Data
df_courses = configure_courses('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/mk_dsi_genap_praktikum.csv')
df_rooms, df_timeslots = configure_rooms_and_timeslots('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/data/rooms.csv')
print(f"Dataset Aktif -> Courses: {len(df_courses)} | Rooms: {len(df_rooms)} | Timeslots: {len(df_timeslots)}")

df_courses.to_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/mk_dsi_genap_praktikum_processed.csv', index=False)
df_rooms.to_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/rooms_processed.csv', index=False)
df_timeslots.to_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/timeslots_processed.csv', index=False)

### Optimized POA from Claude

In [10]:
import numpy as np
import pandas as pd
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
import os

# ==========================================
# 1. Deklarasi & Persiapan Dataset
# ==========================================
print("[INFO] Memuat dan memproses dataset...")
mk_df_raw = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/mk_dsi_genap_praktikum_processed.csv')
rooms_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/rooms_processed.csv')
timeslots_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/timeslots_processed.csv')

# ── Pre-compute valid consecutive timeslot pairs ──────────────────────────────
valid_starts = [
    i for i in range(len(timeslots_df) - 1)
    if (timeslots_df.iloc[i]['day'] == timeslots_df.iloc[i+1]['day'] and
        timeslots_df.iloc[i]['time_end'] == timeslots_df.iloc[i+1]['time_start'])
]

rooms_prac = rooms_df[rooms_df['is_practicum'] == True].index.tolist()
rooms_theo = rooms_df[rooms_df['is_theory'] == True].index.tolist()

# ── Teacher workload heuristic ────────────────────────────────────────────────
teacher_workload = mk_df_raw['teacher_id'].value_counts().to_dict()
mk_df_raw['teacher_degree'] = mk_df_raw['teacher_id'].map(teacher_workload)

# ── Pre-build candidate slot lists PER room type (computed ONCE globally) ─────
# Shape: list of (room_idx, start_timeslot_idx)
PRAC_SLOTS = [(r, s) for r in rooms_prac for s in valid_starts]
THEO_SLOTS = [(r, s) for r in rooms_theo for s in valid_starts]

# Pre-extract arrays for O(1) column access inside hot loop
_REQ_PRAC   = mk_df_raw['requires_practicum_room'].to_numpy()
_TEACHER    = mk_df_raw['teacher_id'].to_numpy()
_T_PRIORITY = mk_df_raw['teacher_priority'].to_numpy()
_T_DEGREE   = mk_df_raw['teacher_degree'].to_numpy()
_TS_DAY     = timeslots_df['day'].to_numpy()
_TS_SESSION = timeslots_df['session_type'].to_numpy()
_N_COURSES  = len(mk_df_raw)

# ── Cache session-type arrays for SC penalty (avoids repeated iloc) ───────────
SESSION_MORNING   = (_TS_SESSION == 'Morning')
SESSION_AFTERNOON = (_TS_SESSION == 'Afternoon')

# ==========================================
# 2. Optimized Fitness Function
# ==========================================
def fitness_function(sol_x):
    """
    Optimizations vs v1:
    - Pre-extracted numpy arrays for column access (no pandas iloc in hot loop)
    - Available-slot SETS updated incrementally (no scan of all R×S per course)
    - Early SC break: skip remaining slot checks once penalty=0 found
    - Vectorized priority sorting with numpy argsort
    """
    hc_value = 0
    sc_value = 0
    hc_count = 0
    sc_count = 0
    assignments = []

    # ── Available-slot sets (copied per call, updated as slots are booked) ────
    avail_prac = set(range(len(PRAC_SLOTS)))
    avail_theo = set(range(len(THEO_SLOTS)))

    # Fast lookup: (room, ts_idx) → booked?
    room_time_booked  = set()
    teacher_time_booked = set()
    teacher_day_rooms = {}   # teacher → {day → [rooms]}

    # ── Decode priority: base heuristic + PO random key ───────────────────────
    base_scores = (_REQ_PRAC.astype(float) * 1000) + (_T_DEGREE * 10)
    priorities  = base_scores + (sol_x * 100)
    queue_order = np.argsort(-priorities)   # descending, no Python sort()

    for idx in queue_order:
        req_prac   = bool(_REQ_PRAC[idx])
        teacher    = _TEACHER[idx]
        t_priority = int(_T_PRIORITY[idx])

        slot_list  = PRAC_SLOTS if req_prac else THEO_SLOTS
        avail_set  = avail_prac  if req_prac else avail_theo

        best_choice    = None
        best_sc_penalty = float('inf')

        # ── Greedy: scan only available (non-booked) slots ───────────────────
        # Iterate over a snapshot of available indices
        for si in list(avail_set):
            r, start_idx = slot_list[si]
            t1, t2 = start_idx, start_idx + 1

            # Hard constraint check  (O(1) set lookup)
            if (r, t1) in room_time_booked or (r, t2) in room_time_booked:
                continue
            if (teacher, t1) in teacher_time_booked or (teacher, t2) in teacher_time_booked:
                continue

            # Soft constraint evaluation
            day   = _TS_DAY[start_idx]
            temp_sc = 0

            if t_priority == 1:
                if not (SESSION_MORNING[t1] and SESSION_MORNING[t2]):
                    temp_sc += 10
            elif t_priority == 2:
                if not (SESSION_AFTERNOON[t1] and SESSION_AFTERNOON[t2]):
                    temp_sc += 10

            if teacher in teacher_day_rooms and day in teacher_day_rooms[teacher]:
                if r not in teacher_day_rooms[teacher][day]:
                    temp_sc += 50

            if temp_sc < best_sc_penalty:
                best_sc_penalty = temp_sc
                best_choice     = (r, start_idx, day, t1, t2, si)
                if temp_sc == 0:
                    break          # Perfect slot found — no need to continue

        # ── Assign or record hard conflict ────────────────────────────────────
        if best_choice is not None:
            r, start_idx, day, t1, t2, si = best_choice

            room_time_booked.add((r, t1));    room_time_booked.add((r, t2))
            teacher_time_booked.add((teacher, t1)); teacher_time_booked.add((teacher, t2))

            # Remove booked room-slots from available sets to prune future searches
            # (keep set accurate: remove any slot that now overlaps)
            target_set = avail_prac if req_prac else avail_theo
            target_list = PRAC_SLOTS if req_prac else THEO_SLOTS
            for si2 in list(target_set):
                r2, s2 = target_list[si2]
                if r2 == r and (s2 == t1 or s2 == t2 or s2 + 1 == t1):
                    target_set.discard(si2)

            sc_value += best_sc_penalty
            assignments.append({
                'course_i': int(idx), 'room': r,
                'start_slot': start_idx, 'end_slot': t2,
                'teacher': teacher, 'priority': t_priority, 'day': day
            })

            if teacher not in teacher_day_rooms:
                teacher_day_rooms[teacher] = {}
            if day not in teacher_day_rooms[teacher]:
                teacher_day_rooms[teacher][day] = []
            teacher_day_rooms[teacher][day].append(r)

        else:
            hc_count += 1
            hc_value += 100_000

    # ── Re-evaluate room-switching SC ─────────────────────────────────────────
    for teacher, days in teacher_day_rooms.items():
        for day, rooms in days.items():
            unique_rooms = len(set(rooms))
            if unique_rooms > 1:
                v = unique_rooms - 1
                sc_count += v
                sc_value += 50 * v

    total_penalty = hc_value + sc_value
    metrics = {
        'hc_count': hc_count, 'hc_value': hc_value,
        'sc_count': sc_count, 'sc_value': sc_value
    }
    return total_penalty, assignments, metrics


# ── Wrapper for parallel worker (top-level so it's picklable) ─────────────────
def _eval_worker(x):
    return fitness_function(x)


# ==========================================
# 3. Solution Class
# ==========================================
class Solution:
    __slots__ = ('X', 'Cost', 'assignments', 'metrics')

    def __init__(self, x, cost, assignments=None, metrics=None):
        self.X           = x
        self.Cost        = cost
        self.assignments = assignments
        self.metrics     = metrics

    def clone(self):
        return Solution(self.X.copy(), self.Cost, self.assignments, self.metrics)


# ==========================================
# 4. Parallel Population Evaluator
# ==========================================
def parallel_eval(vectors, n_workers=None):
    """Evaluate a list of numpy vectors in parallel. Falls back to serial if n_workers=1."""
    n_workers = n_workers or min(os.cpu_count(), len(vectors))
    if n_workers <= 1 or len(vectors) <= 4:
        return [fitness_function(v) for v in vectors]

    results = [None] * len(vectors)
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_eval_worker, v): i for i, v in enumerate(vectors)}
        for fut in as_completed(futures):
            results[futures[fut]] = fut.result()
    return results


# ==========================================
# 5. Puma Optimizer – Optimized
# ==========================================
def print_iteration_log(Iter, Sol, Best, elapsed):
    worst_cost = max(s.Cost for s in Sol)
    m = Best.metrics
    print(
        f"Iter {Iter:03d} | Best: {Best.Cost:8.0f} | Worst: {worst_cost:8.0f} | "
        f"HC: {m['hc_count']:3d} ({m['hc_value']:7.0f}) | SC: {m['sc_value']:5.0f} | "
        f"Time: {elapsed:.1f}s"
    )


def exploration(Sol, lb, ub, dim, nSol, n_workers):
    pCR = 0.20
    PCR = 1 - pCR
    p   = PCR / nSol
    Sol.sort(key=lambda s: s.Cost)

    new_vectors = []
    for i in range(nSol):
        x = Sol[i].X
        A = list(range(nSol)); A.remove(i); np.random.shuffle(A)
        a, b, c, d, e, f = A[:6]
        G = 2 * np.random.rand() - 1
        if np.random.rand() < 0.5:
            y = np.random.rand(dim) * (ub - lb) + lb
        else:
            da = Sol[a].X - Sol[b].X
            db = Sol[c].X - Sol[d].X
            dc = Sol[e].X - Sol[f].X
            y  = Sol[a].X + G * da + G * ((da - db) + (db - dc))
        y  = np.clip(y, lb, ub)
        z  = x.copy()
        j0 = np.random.randint(dim)
        mask = (np.random.rand(dim) <= pCR)
        mask[j0] = True
        z[mask] = y[mask]
        new_vectors.append(z)

    results = parallel_eval(new_vectors, n_workers)
    for i, (cost, asg, met) in enumerate(results):
        candidate = Solution(new_vectors[i], cost, asg, met)
        if candidate.Cost < Sol[i].Cost:
            Sol[i] = candidate
        else:
            pCR = pCR + p
    return Sol


def exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers):
    Q, Beta = 0.67, 2
    mbest = np.mean([s.X for s in Sol], axis=0)

    new_vectors = []
    for i in range(nSol):
        beta1 = 2 * np.random.rand()
        beta2 = np.random.randn(dim)
        w, v  = np.random.randn(dim), np.random.randn(dim)
        F1    = np.random.randn(dim) * np.exp(2 - Iter * (2 / MaxIter))
        F2    = w * (v ** 2) * np.cos(2 * np.random.rand() * w)
        R_1   = 2 * np.random.rand() - 1
        S1    = (2 * np.random.rand() - 1 + np.random.randn(dim))
        S2    = F1 * R_1 * Sol[i].X + F2 * (1 - R_1) * Best.X
        VEC   = S2 / (S1 + 1e-10)

        if np.random.rand() <= 0.5:
            if np.random.rand() > Q:
                r_idx = np.random.randint(nSol)
                x_new = Best.X + beta1 * np.exp(beta2) * (Sol[r_idx].X - Sol[i].X)
            else:
                x_new = beta1 * VEC - Best.X
        else:
            r1   = np.random.randint(nSol)
            sign = (-1) ** np.random.randint(0, 2)
            x_new = (mbest * Sol[r1].X - sign * Sol[i].X) / (1 + Beta * np.random.rand())

        new_vectors.append(np.clip(x_new, lb, ub))

    results = parallel_eval(new_vectors, n_workers)
    for i, (cost, asg, met) in enumerate(results):
        candidate = Solution(new_vectors[i], cost, asg, met)
        if candidate.Cost < Sol[i].Cost:
            Sol[i] = candidate
    return Sol


def run_puma_optimizer(nSol, MaxIter, lb, ub, dim, n_workers=None):
    n_workers = n_workers or min(os.cpu_count(), nSol)
    print(f"[INFO] Parallel workers: {n_workers}")

    # ── Initial population ────────────────────────────────────────────────────
    print("Mengevaluasi populasi inisial...")
    init_vectors = [lb + (ub - lb) * np.random.rand(dim) for _ in range(nSol)]

    # Seed one individual with pure-heuristic order (all X=0.5 → rely on base score)
    init_vectors[0] = np.full(dim, 0.5)

    results = parallel_eval(init_vectors, n_workers)
    Sol  = [Solution(init_vectors[i], *results[i]) for i in range(nSol)]
    Best = min(Sol, key=lambda s: s.Cost)
    Initial_Best = Best.clone()

    print(f"\nInitial Best Fitness: {Best.Cost:.0f} | "
          f"HC: {Best.metrics['hc_count']} | SC: {Best.metrics['sc_value']:.0f}")

    # ── Early exit if already zero conflicts ──────────────────────────────────
    if Best.metrics['hc_count'] == 0:
        print("[INFO] Zero HC violations in initial population — skipping optimization!")
        return Best

    UnSelected = [1, 1]
    F3_Explore = F3_Exploit = 0
    Seq_Time_Explore  = [1, 1, 1]; Seq_Time_Exploit  = [1, 1, 1]
    Seq_Cost_Explore  = [1, 1, 1]; Seq_Cost_Exploit  = [1, 1, 1]
    PF     = [0.5, 0.5, 0.3]
    PF_F3  = []
    Mega_Explor = Mega_Exploit = 0.99
    Costs_Explor  = np.zeros(3)
    Costs_Exploit = np.zeros(3)
    Flag_Change   = 1

    print("\n" + "=" * 95)
    print("MEMULAI PROSES OPTIMASI PUMA (v2)".center(95))
    print("=" * 95)

    t_start = time.perf_counter()

    # ── Bootstrap: first 3 iterations (both explore + exploit to seed PF) ────
    for Iter in range(1, 4):
        t0 = time.perf_counter()
        Sol_E1 = exploration ([s.clone() for s in Sol], lb, ub, dim, nSol, n_workers)
        Sol_E2 = exploitation([s.clone() for s in Sol], lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers)
        Costs_Explor[Iter-1]  = min(s.Cost for s in Sol_E1)
        Costs_Exploit[Iter-1] = min(s.Cost for s in Sol_E2)

        # Keep top nSol across all candidates
        merged = Sol + Sol_E1 + Sol_E2
        merged.sort(key=lambda s: s.Cost)
        Sol  = merged[:nSol]
        Best = Sol[0]
        print_iteration_log(Iter, Sol, Best, time.perf_counter() - t_start)

        # ── Early exit ────────────────────────────────────────────────────────
        if Best.metrics['hc_count'] == 0:
            print("[INFO] Zero HC achieved — stopping early.")
            return Best

    Seq_Cost_Explore[0] = abs(Initial_Best.Cost - Costs_Explor[0])
    Seq_Cost_Explore[1] = abs(Costs_Explor[1]  - Costs_Explor[0])
    Seq_Cost_Explore[2] = abs(Costs_Explor[2]  - Costs_Explor[1])
    Seq_Cost_Exploit[0] = abs(Initial_Best.Cost - Costs_Exploit[0])
    Seq_Cost_Exploit[1] = abs(Costs_Exploit[1] - Costs_Exploit[0])
    Seq_Cost_Exploit[2] = abs(Costs_Exploit[2] - Costs_Exploit[1])

    for v in Seq_Cost_Explore + Seq_Cost_Exploit:
        if v != 0: PF_F3.append(v)

    # ── Main loop ─────────────────────────────────────────────────────────────
    stagnation = 0
    prev_best  = Best.Cost

    for Iter in range(4, MaxIter + 1):
        F1_Explor  = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0])
        F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
        F2_Explor  = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore))
        F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
        Score_Explore  = Mega_Explor  * (F1_Explor  + F2_Explor)
        Score_Exploit  = Mega_Exploit * (F1_Exploit + F2_Exploit)

        if Score_Explore > Score_Exploit:
            SelectFlag = 1
            Sol = exploration(Sol, lb, ub, dim, nSol, n_workers)
            Count_select = UnSelected.copy()
            UnSelected[1] += 1; UnSelected[0] = 1
            F3_Explore = PF[2]; F3_Exploit += PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Explore[2], Seq_Cost_Explore[1] = Seq_Cost_Explore[1], Seq_Cost_Explore[0]
            Seq_Cost_Explore[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Explore[0]: PF_F3.append(Seq_Cost_Explore[0])
        else:
            SelectFlag = 2
            Sol = exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers)
            Count_select = UnSelected.copy()
            UnSelected[0] += 1; UnSelected[1] = 1
            F3_Explore += PF[2]; F3_Exploit = PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Exploit[2], Seq_Cost_Exploit[1] = Seq_Cost_Exploit[1], Seq_Cost_Exploit[0]
            Seq_Cost_Exploit[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Exploit[0]: PF_F3.append(Seq_Cost_Exploit[0])

        if TBest.Cost < Best.Cost:
            Best = TBest

        if Flag_Change != SelectFlag:
            Flag_Change = SelectFlag
            Seq_Time_Explore[2], Seq_Time_Explore[1], Seq_Time_Explore[0] = \
                Seq_Time_Explore[1], Seq_Time_Explore[0], Count_select[0]
            Seq_Time_Exploit[2], Seq_Time_Exploit[1], Seq_Time_Exploit[0] = \
                Seq_Time_Exploit[1], Seq_Time_Exploit[0], Count_select[1]

        if Score_Explore < Score_Exploit:
            Mega_Explor  = max(Mega_Explor  - 0.01, 0.01); Mega_Exploit = 0.99
        else:
            Mega_Explor  = 0.99; Mega_Exploit = max(Mega_Exploit - 0.01, 0.01)

        print_iteration_log(Iter, Sol, Best, time.perf_counter() - t_start)

        # ── Early exit on zero HC ─────────────────────────────────────────────
        if Best.metrics['hc_count'] == 0:
            print(f"[INFO] Zero HC violations achieved at iteration {Iter} — stopping early.")
            break

        # ── Stagnation restart (inject random diversity after 10 flat iters) ──
        if Best.Cost >= prev_best:
            stagnation += 1
        else:
            stagnation = 0
            prev_best  = Best.Cost

        if stagnation >= 10:
            print(f"[WARN] Stagnation detected at iter {Iter} — injecting diversity...")
            n_replace = nSol // 3
            for k in range(1, n_replace + 1):
                x_new = lb + (ub - lb) * np.random.rand(dim)
                c, a, m = fitness_function(x_new)
                Sol[-k] = Solution(x_new, c, a, m)
            stagnation = 0

    total_time = time.perf_counter() - t_start
    print(f"\n[INFO] Optimasi selesai dalam {total_time:.1f}s")
    return Best


# ==========================================
# 6. Eksekusi & Ekspor
# ==========================================
if __name__ == '__main__':
    nSol    = 100
    MaxIter = 100
    dim     = _N_COURSES
    lb, ub  = 0.0, 1.0

    t0 = time.perf_counter()
    best_solution = run_puma_optimizer(nSol, MaxIter, lb, ub, dim, n_workers=None)
    print(f"\nTotal elapsed: {time.perf_counter() - t0:.1f}s")

    # ── Build result DataFrame ────────────────────────────────────────────────
    res_list = []
    for a in best_solution.assignments:
        course = mk_df_raw.iloc[a['course_i']]
        room   = rooms_df.iloc[a['room']]
        t1     = timeslots_df.iloc[a['start_slot']]
        t2     = timeslots_df.iloc[a['end_slot']]
        res_list.append({
            'event_id':    course['event_id'],
            'course_name': course['course_name'],
            'teacher_id':  course['teacher_id'],
            'room_id':     room['room_id'],
            'day':         t1['day'],
            'time_start':  t1['time_start'],
            'time_end':    t2['time_end'],
        })

    res_df = pd.DataFrame(res_list)
    out_path = 'scheduled_courses_poa_v2.csv'
    res_df.to_csv(out_path, index=False)

    m = best_solution.metrics
    print(f"\n{'='*60}")
    print(f"  FINAL RESULT")
    print(f"  HC Violations : {m['hc_count']}  (penalty: {m['hc_value']})")
    print(f"  SC Violations : {m['sc_count']}  (penalty: {m['sc_value']})")
    print(f"  Total Fitness : {best_solution.Cost}")
    print(f"  Courses scheduled: {len(res_list)} / {_N_COURSES}")
    print(f"{'='*60}")
    print(f"[INFO] Solusi disimpan di '{out_path}'")

### POA by Claude with Repaired Mechanism

In [12]:
import numpy as np
import pandas as pd
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
import os

# ==========================================
# 1. Deklarasi & Persiapan Dataset
# ==========================================
print("[INFO] Memuat dan memproses dataset...")
mk_df_raw = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/mk_dsi_genap_praktikum_processed.csv')
rooms_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/rooms_processed.csv')
timeslots_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/timeslots_processed.csv')

# ── Pre-compute valid consecutive timeslot pairs ──────────────────────────────
valid_starts = [
    i for i in range(len(timeslots_df) - 1)
    if (timeslots_df.iloc[i]['day'] == timeslots_df.iloc[i + 1]['day'] and
        timeslots_df.iloc[i]['time_end'] == timeslots_df.iloc[i + 1]['time_start'])
]

rooms_prac = rooms_df[rooms_df['is_practicum'] == True].index.tolist()
rooms_theo = rooms_df[rooms_df['is_theory'] == True].index.tolist()

# ── Teacher workload heuristic ────────────────────────────────────────────────
teacher_workload = mk_df_raw['teacher_id'].value_counts().to_dict()
mk_df_raw['teacher_degree'] = mk_df_raw['teacher_id'].map(teacher_workload)

# ── Pre-build global slot lists (computed ONCE) ───────────────────────────────
PRAC_SLOTS = [(r, s) for r in rooms_prac for s in valid_starts]
THEO_SLOTS = [(r, s) for r in rooms_theo for s in valid_starts]

# ── Pre-extracted numpy arrays for O(1) hot-loop access ──────────────────────
_REQ_PRAC   = mk_df_raw['requires_practicum_room'].to_numpy()
_TEACHER    = mk_df_raw['teacher_id'].to_numpy()
_T_PRIORITY = mk_df_raw['teacher_priority'].to_numpy()
_T_DEGREE   = mk_df_raw['teacher_degree'].to_numpy()
_TS_DAY     = timeslots_df['day'].to_numpy()
_TS_SESSION = timeslots_df['session_type'].to_numpy()
_N_COURSES  = len(mk_df_raw)

SESSION_MORNING   = (_TS_SESSION == 'Morning')
SESSION_AFTERNOON = (_TS_SESSION == 'Afternoon')

# ── HC weight: any 1 HC violation > any possible SC total ────────────────────
HC_WEIGHT = 10_000_000


# ==========================================
# 2. Fitness Function with Conflict-Directed Repair
# ==========================================
def fitness_function(sol_x):
    """
    Three-phase decoder:

    PHASE 1 — Greedy HC-aware assignment
        Courses ordered by Most-Constrained-First + PO random key.
        For each course, find the best HC-free slot (with bounded SC scan).
        Unassigned courses are collected for repair rather than immediately
        counted as violations.

    PHASE 2 — Conflict-Directed Repair
        For each unassigned course (no HC-free slot found in Phase 1):
          1. Scan ALL slots of the correct room type.
          2. For each slot blocked only by a TEACHER conflict (room is free):
             - Identify the "blocker" assignment occupying that teacher-timeslot.
             - Try to relocate the blocker to any other HC-free slot.
             - If relocation succeeds → free the slot, assign the waiting course.
          3. Try up to _MAX_REPAIR_ATTEMPTS relocations per unassigned course.
        This directly resolves the root cause: courses stuck because a
        previously placed course holds the only viable teacher-timeslot.

    PHASE 3 — SC Re-evaluation
        Full SC recalculation after all assignments (including repairs) are final.

    FITNESS = hc_count × HC_WEIGHT + sc_value
    """
    hc_count = 0
    sc_value = 0
    sc_count = 0
    assignments     = []    # list of assigned course dicts
    unassigned_idxs = []    # course indices that Phase 1 could not place

    # Per-call booking state
    room_time_booked      = set()   # (room_idx, ts_idx)
    teacher_time_booked   = set()   # (teacher_id, ts_idx)
    teacher_day_rooms     = {}      # teacher → {day → [room, ...]}

    # Reverse index: (teacher_id, ts_idx) → assignment list index
    # Used in Phase 2 to quickly find which assignment blocks a slot
    teacher_ts_to_asgn = {}         # (teacher, ts_idx) → index in `assignments`

    # Available-slot sets (pruned as rooms get booked)
    avail_prac = set(range(len(PRAC_SLOTS)))
    avail_theo = set(range(len(THEO_SLOTS)))

    # ── MCF + PO random-key ordering ─────────────────────────────────────────
    base_scores = (_REQ_PRAC.astype(float) * 1000) + (_T_DEGREE * 10)
    priorities  = base_scores + (sol_x * 100)
    queue_order = np.argsort(-priorities)

    # =========================================================================
    # PHASE 1 — GREEDY ASSIGNMENT
    # =========================================================================
    for idx in queue_order:
        req_prac   = bool(_REQ_PRAC[idx])
        teacher    = _TEACHER[idx]
        t_priority = int(_T_PRIORITY[idx])

        slot_list = PRAC_SLOTS if req_prac else THEO_SLOTS
        avail_set = avail_prac  if req_prac else avail_theo

        hc_fallback    = None      # first HC-free slot (guaranteed commit)
        best_sc        = float('inf')
        best_slot_info = None
        sc_scan_count  = 0

        for si in list(avail_set):
            r, s = slot_list[si]
            t1, t2 = s, s + 1

            # Teacher conflict check (room already filtered by avail_set)
            if (teacher, t1) in teacher_time_booked or (teacher, t2) in teacher_time_booked:
                continue

            # First HC-free slot → save as fallback
            if hc_fallback is None:
                hc_fallback = (r, s, _TS_DAY[s], t1, t2, si)

            # Bounded SC evaluation
            day    = _TS_DAY[s]
            temp_sc = 0
            if t_priority == 1 and not (SESSION_MORNING[t1] and SESSION_MORNING[t2]):
                temp_sc += 10
            elif t_priority == 2 and not (SESSION_AFTERNOON[t1] and SESSION_AFTERNOON[t2]):
                temp_sc += 10
            if teacher in teacher_day_rooms and day in teacher_day_rooms[teacher]:
                if r not in teacher_day_rooms[teacher][day]:
                    temp_sc += 50

            if temp_sc < best_sc:
                best_sc        = temp_sc
                best_slot_info = (r, s, day, t1, t2, si)
                if temp_sc == 0:
                    break
            sc_scan_count += 1
            if sc_scan_count >= 50:
                break

        chosen = best_slot_info if best_slot_info is not None else hc_fallback

        if chosen is not None:
            r, s, day, t1, t2, si = chosen
            _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
                       assignments, room_time_booked, teacher_time_booked,
                       teacher_day_rooms, teacher_ts_to_asgn,
                       avail_prac, avail_theo)
        else:
            # No HC-free slot — queue for repair
            unassigned_idxs.append(int(idx))

    # =========================================================================
    # PHASE 2 — CONFLICT-DIRECTED REPAIR
    # Attempt to place every unassigned course by relocating its blocker.
    # =========================================================================
    still_unassigned = []

    for idx in unassigned_idxs:
        req_prac   = bool(_REQ_PRAC[idx])
        teacher    = _TEACHER[idx]
        t_priority = int(_T_PRIORITY[idx])

        slot_list = PRAC_SLOTS if req_prac else THEO_SLOTS
        avail_set = avail_prac  if req_prac else avail_theo

        placed = False

        # Scan ALL slots of the correct room type (not just avail_set —
        # we need to inspect booked slots to find swappable blockers)
        for r, s in slot_list:
            t1, t2 = s, s + 1

            # Room must be free (we can only swap teacher conflicts, not room conflicts)
            if (r, t1) in room_time_booked or (r, t2) in room_time_booked:
                continue

            # Check which timeslot the teacher is blocked on
            blocked_t1 = (teacher, t1) in teacher_time_booked
            blocked_t2 = (teacher, t2) in teacher_time_booked

            if not blocked_t1 and not blocked_t2:
                # Slot is actually free — assign directly (safety net)
                day = _TS_DAY[s]
                _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
                           assignments, room_time_booked, teacher_time_booked,
                           teacher_day_rooms, teacher_ts_to_asgn,
                           avail_prac, avail_theo)
                placed = True
                break

            # Identify the blocker assignment(s) for this teacher at t1 or t2
            blocker_key = (teacher, t1) if blocked_t1 else (teacher, t2)
            blocker_asgn_idx = teacher_ts_to_asgn.get(blocker_key)
            if blocker_asgn_idx is None:
                continue

            blocker = assignments[blocker_asgn_idx]
            b_idx       = blocker['course_i']
            b_req_prac  = bool(_REQ_PRAC[b_idx])
            b_teacher   = blocker['teacher']
            b_t1_old    = blocker['start_slot']
            b_t2_old    = blocker['end_slot']
            b_r_old     = blocker['room']
            b_priority  = blocker['priority']

            # Temporarily unbook the blocker
            room_time_booked.discard((b_r_old, b_t1_old))
            room_time_booked.discard((b_r_old, b_t2_old))
            teacher_time_booked.discard((b_teacher, b_t1_old))
            teacher_time_booked.discard((b_teacher, b_t2_old))
            teacher_ts_to_asgn.pop((b_teacher, b_t1_old), None)
            teacher_ts_to_asgn.pop((b_teacher, b_t2_old), None)

            # Also remove from teacher_day_rooms
            b_day_old = blocker['day']
            if b_teacher in teacher_day_rooms and b_day_old in teacher_day_rooms[b_teacher]:
                try:
                    teacher_day_rooms[b_teacher][b_day_old].remove(b_r_old)
                except ValueError:
                    pass
                if not teacher_day_rooms[b_teacher][b_day_old]:
                    del teacher_day_rooms[b_teacher][b_day_old]

            # Re-add blocker's old room-slot to avail sets so it can be rescanned
            b_slot_list = PRAC_SLOTS if b_req_prac else THEO_SLOTS
            b_avail_set = avail_prac  if b_req_prac else avail_theo
            try:
                b_si = b_slot_list.index((b_r_old, b_t1_old))
                b_avail_set.add(b_si)
            except ValueError:
                pass

            # Try to find a new slot for the blocker
            reloc_found = None
            b_avail     = avail_prac if b_req_prac else avail_theo
            b_slots     = PRAC_SLOTS if b_req_prac else THEO_SLOTS
            for si2 in list(b_avail):
                r2, s2 = b_slots[si2]
                t1n, t2n = s2, s2 + 1
                if (r2, t1n) in room_time_booked or (r2, t2n) in room_time_booked:
                    continue
                if (b_teacher, t1n) in teacher_time_booked or (b_teacher, t2n) in teacher_time_booked:
                    continue
                # Valid relocation slot found
                reloc_found = (r2, s2, _TS_DAY[s2], t1n, t2n, si2)
                break

            if reloc_found is not None:
                # Commit blocker to its new slot
                r2, s2, day2, t1n, t2n, si2 = reloc_found
                _do_assign(b_idx, r2, s2, day2, t1n, t2n, b_priority, b_teacher,
                           b_req_prac, assignments, room_time_booked,
                           teacher_time_booked, teacher_day_rooms,
                           teacher_ts_to_asgn, avail_prac, avail_theo,
                           update_idx=blocker_asgn_idx)

                # Now assign the waiting course to the freed slot
                day = _TS_DAY[s]
                _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
                           assignments, room_time_booked, teacher_time_booked,
                           teacher_day_rooms, teacher_ts_to_asgn,
                           avail_prac, avail_theo)
                placed = True
                break
            else:
                # Relocation failed — restore the blocker to its original slot
                room_time_booked.add((b_r_old, b_t1_old))
                room_time_booked.add((b_r_old, b_t2_old))
                teacher_time_booked.add((b_teacher, b_t1_old))
                teacher_time_booked.add((b_teacher, b_t2_old))
                teacher_ts_to_asgn[(b_teacher, b_t1_old)] = blocker_asgn_idx
                teacher_ts_to_asgn[(b_teacher, b_t2_old)] = blocker_asgn_idx
                if b_teacher not in teacher_day_rooms:
                    teacher_day_rooms[b_teacher] = {}
                if b_day_old not in teacher_day_rooms[b_teacher]:
                    teacher_day_rooms[b_teacher][b_day_old] = []
                teacher_day_rooms[b_teacher][b_day_old].append(b_r_old)
                # Remove the slot we temporarily re-added
                b_avail_set.discard(b_si if 'b_si' in dir() else -1)

        if not placed:
            still_unassigned.append(idx)

    hc_count = len(still_unassigned)

    # =========================================================================
    # PHASE 3 — SC TOTAL EVALUATION
    # =========================================================================
    for asgn in assignments:
        t1 = asgn['start_slot']
        t2 = asgn['end_slot']
        tp = asgn['priority']
        if tp == 1 and not (SESSION_MORNING[t1] and SESSION_MORNING[t2]):
            sc_value += 10
        elif tp == 2 and not (SESSION_AFTERNOON[t1] and SESSION_AFTERNOON[t2]):
            sc_value += 10

    for teacher_id, days in teacher_day_rooms.items():
        for day, rooms in days.items():
            unique_rooms = len(set(rooms))
            if unique_rooms > 1:
                v = unique_rooms - 1
                sc_count += v
                sc_value += 50 * v

    fitness = hc_count * HC_WEIGHT + sc_value
    metrics = {
        'hc_count': hc_count,
        'hc_value': hc_count * HC_WEIGHT,
        'sc_count': sc_count,
        'sc_value': sc_value,
        'repaired': len(unassigned_idxs) - hc_count,   # how many Phase 2 fixed
    }
    return fitness, assignments, metrics


# ==========================================
# 3. Assignment Helper (DRY – used by both phases)
# ==========================================
def _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
               assignments, room_time_booked, teacher_time_booked,
               teacher_day_rooms, teacher_ts_to_asgn,
               avail_prac, avail_theo, update_idx=None):
    """Book a slot and record the assignment.
    If update_idx is given, update an existing assignment in-place (for repair swaps).
    """
    room_time_booked.add((r, t1));      room_time_booked.add((r, t2))
    teacher_time_booked.add((teacher, t1)); teacher_time_booked.add((teacher, t2))

    # Prune available sets for this room
    slot_list = PRAC_SLOTS if req_prac else THEO_SLOTS
    avail_set = avail_prac  if req_prac else avail_theo
    for si2 in list(avail_set):
        r2, s2 = slot_list[si2]
        if r2 == r and (s2 == t1 or s2 == t2 or s2 + 1 == t1):
            avail_set.discard(si2)

    if teacher not in teacher_day_rooms:
        teacher_day_rooms[teacher] = {}
    if day not in teacher_day_rooms[teacher]:
        teacher_day_rooms[teacher][day] = []
    teacher_day_rooms[teacher][day].append(r)

    asgn = {
        'course_i': int(idx), 'room': r,
        'start_slot': s, 'end_slot': t2,
        'teacher': teacher, 'priority': t_priority, 'day': day
    }

    if update_idx is not None:
        assignments[update_idx] = asgn
        teacher_ts_to_asgn[(teacher, t1)] = update_idx
        teacher_ts_to_asgn[(teacher, t2)] = update_idx
    else:
        new_idx = len(assignments)
        assignments.append(asgn)
        teacher_ts_to_asgn[(teacher, t1)] = new_idx
        teacher_ts_to_asgn[(teacher, t2)] = new_idx


# ── Parallel worker wrapper ────────────────────────────────────────────────────
def _eval_worker(x):
    return fitness_function(x)


# ==========================================
# 4. Solution Class
# ==========================================
class Solution:
    __slots__ = ('X', 'Cost', 'assignments', 'metrics')

    def __init__(self, x, cost, assignments=None, metrics=None):
        self.X           = x
        self.Cost        = cost
        self.assignments = assignments
        self.metrics     = metrics

    def clone(self):
        return Solution(self.X.copy(), self.Cost, self.assignments, self.metrics)


# ==========================================
# 5. Parallel Population Evaluator
# ==========================================
def parallel_eval(vectors, n_workers=None):
    n_workers = n_workers or min(os.cpu_count(), len(vectors))
    if n_workers <= 1 or len(vectors) <= 4:
        return [fitness_function(v) for v in vectors]
    results = [None] * len(vectors)
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_eval_worker, v): i for i, v in enumerate(vectors)}
        for fut in as_completed(futures):
            results[futures[fut]] = fut.result()
    return results


# ==========================================
# 6. Puma Optimizer — MECHANISM UNCHANGED
# ==========================================
def print_iteration_log(Iter, Sol, Best, elapsed):
    worst_cost = max(s.Cost for s in Sol)
    m = Best.metrics
    repaired = m.get('repaired', 0)
    print(
        f"Iter {Iter:03d} | Best: {Best.Cost:12.0f} | Worst: {worst_cost:12.0f} | "
        f"HC: {m['hc_count']:3d} | Repaired: {repaired:3d} | "
        f"SC: {m['sc_value']:6.0f} | {elapsed:.1f}s"
    )


def exploration(Sol, lb, ub, dim, nSol, n_workers):
    pCR = 0.20
    PCR = 1 - pCR
    p   = PCR / nSol
    Sol.sort(key=lambda s: s.Cost)

    new_vectors = []
    for i in range(nSol):
        x = Sol[i].X
        A = list(range(nSol)); A.remove(i); np.random.shuffle(A)
        a, b, c, d, e, f = A[:6]
        G = 2 * np.random.rand() - 1
        if np.random.rand() < 0.5:
            y = np.random.rand(dim) * (ub - lb) + lb
        else:
            da = Sol[a].X - Sol[b].X
            db = Sol[c].X - Sol[d].X
            dc = Sol[e].X - Sol[f].X
            y  = Sol[a].X + G * da + G * ((da - db) + (db - dc))
        y  = np.clip(y, lb, ub)
        z  = x.copy()
        j0 = np.random.randint(dim)
        mask = (np.random.rand(dim) <= pCR)
        mask[j0] = True
        z[mask] = y[mask]
        new_vectors.append(z)

    results = parallel_eval(new_vectors, n_workers)
    for i, (cost, asg, met) in enumerate(results):
        candidate = Solution(new_vectors[i], cost, asg, met)
        if candidate.Cost < Sol[i].Cost:
            Sol[i] = candidate
        else:
            pCR = pCR + p
    return Sol


def exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers):
    Q, Beta = 0.67, 2
    mbest   = np.mean([s.X for s in Sol], axis=0)

    new_vectors = []
    for i in range(nSol):
        beta1 = 2 * np.random.rand()
        beta2 = np.random.randn(dim)
        w, v  = np.random.randn(dim), np.random.randn(dim)
        F1    = np.random.randn(dim) * np.exp(2 - Iter * (2 / MaxIter))
        F2    = w * (v ** 2) * np.cos(2 * np.random.rand() * w)
        R_1   = 2 * np.random.rand() - 1
        S1    = (2 * np.random.rand() - 1 + np.random.randn(dim))
        S2    = F1 * R_1 * Sol[i].X + F2 * (1 - R_1) * Best.X
        VEC   = S2 / (S1 + 1e-10)

        if np.random.rand() <= 0.5:
            if np.random.rand() > Q:
                r_idx = np.random.randint(nSol)
                x_new = Best.X + beta1 * np.exp(beta2) * (Sol[r_idx].X - Sol[i].X)
            else:
                x_new = beta1 * VEC - Best.X
        else:
            r1    = np.random.randint(nSol)
            sign  = (-1) ** np.random.randint(0, 2)
            x_new = (mbest * Sol[r1].X - sign * Sol[i].X) / (1 + Beta * np.random.rand())

        new_vectors.append(np.clip(x_new, lb, ub))

    results = parallel_eval(new_vectors, n_workers)
    for i, (cost, asg, met) in enumerate(results):
        candidate = Solution(new_vectors[i], cost, asg, met)
        if candidate.Cost < Sol[i].Cost:
            Sol[i] = candidate
    return Sol


def run_puma_optimizer(nSol, MaxIter, lb, ub, dim, n_workers=None):
    n_workers = n_workers or min(os.cpu_count(), nSol)
    print(f"[INFO] Parallel workers : {n_workers}")
    print(f"[INFO] HC weight        : {HC_WEIGHT:,}")
    print(f"[INFO] Total courses    : {_N_COURSES}")

    # ── Initial population ────────────────────────────────────────────────────
    print("\nMengevaluasi populasi inisial...")
    init_vectors = [lb + (ub - lb) * np.random.rand(dim) for _ in range(nSol)]
    init_vectors[0] = np.full(dim, 0.5)   # Seed: pure MCF order

    results      = parallel_eval(init_vectors, n_workers)
    Sol          = [Solution(init_vectors[i], *results[i]) for i in range(nSol)]
    Best         = min(Sol, key=lambda s: s.Cost)
    Initial_Best = Best.clone()

    m = Best.metrics
    print(f"\nInitial Best  | Fitness: {Best.Cost:12.0f} | "
          f"HC: {m['hc_count']:3d} | Repaired: {m.get('repaired',0):3d} | "
          f"SC: {m['sc_value']:.0f}")

    if Best.metrics['hc_count'] == 0:
        print("[INFO] Zero HC violations at initialisation — skipping optimisation.")
        return Best

    # ── PO state variables (UNCHANGED) ───────────────────────────────────────
    UnSelected = [1, 1]
    F3_Explore = F3_Exploit = 0
    Seq_Time_Explore  = [1, 1, 1];  Seq_Time_Exploit  = [1, 1, 1]
    Seq_Cost_Explore  = [1, 1, 1];  Seq_Cost_Exploit  = [1, 1, 1]
    PF            = [0.5, 0.5, 0.3]
    PF_F3         = []
    Mega_Explor   = Mega_Exploit = 0.99
    Costs_Explor  = np.zeros(3)
    Costs_Exploit = np.zeros(3)
    Flag_Change   = 1

    print("\n" + "=" * 100)
    print("MEMULAI PROSES OPTIMASI PUMA v4 — CONFLICT-DIRECTED REPAIR".center(100))
    print("=" * 100)

    t_start = time.perf_counter()

    # ── Bootstrap iterations 1–3 ──────────────────────────────────────────────
    for Iter in range(1, 4):
        Sol_E1 = exploration ([s.clone() for s in Sol], lb, ub, dim, nSol, n_workers)
        Sol_E2 = exploitation([s.clone() for s in Sol], lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers)
        Costs_Explor[Iter - 1]  = min(s.Cost for s in Sol_E1)
        Costs_Exploit[Iter - 1] = min(s.Cost for s in Sol_E2)

        merged = Sol + Sol_E1 + Sol_E2
        merged.sort(key=lambda s: s.Cost)
        Sol  = merged[:nSol]
        Best = Sol[0]
        print_iteration_log(Iter, Sol, Best, time.perf_counter() - t_start)

        if Best.metrics['hc_count'] == 0:
            print(f"[INFO] Zero HC achieved at bootstrap iteration {Iter}.")
            return Best

    Seq_Cost_Explore[0] = abs(Initial_Best.Cost  - Costs_Explor[0])
    Seq_Cost_Explore[1] = abs(Costs_Explor[1]    - Costs_Explor[0])
    Seq_Cost_Explore[2] = abs(Costs_Explor[2]    - Costs_Explor[1])
    Seq_Cost_Exploit[0] = abs(Initial_Best.Cost  - Costs_Exploit[0])
    Seq_Cost_Exploit[1] = abs(Costs_Exploit[1]   - Costs_Exploit[0])
    Seq_Cost_Exploit[2] = abs(Costs_Exploit[2]   - Costs_Exploit[1])

    for v in Seq_Cost_Explore + Seq_Cost_Exploit:
        if v != 0:
            PF_F3.append(v)

    # ── Main loop (PO mechanism UNCHANGED) ───────────────────────────────────
    stagnation = 0
    prev_best  = Best.Cost

    for Iter in range(4, MaxIter + 1):
        F1_Explor  = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0])
        F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
        F2_Explor  = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore))
        F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
        Score_Explore  = Mega_Explor  * (F1_Explor  + F2_Explor)
        Score_Exploit  = Mega_Exploit * (F1_Exploit + F2_Exploit)

        if Score_Explore > Score_Exploit:
            SelectFlag = 1
            Sol = exploration(Sol, lb, ub, dim, nSol, n_workers)
            Count_select = UnSelected.copy()
            UnSelected[1] += 1;  UnSelected[0] = 1
            F3_Explore = PF[2];  F3_Exploit += PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Explore[2], Seq_Cost_Explore[1] = Seq_Cost_Explore[1], Seq_Cost_Explore[0]
            Seq_Cost_Explore[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Explore[0]:
                PF_F3.append(Seq_Cost_Explore[0])
        else:
            SelectFlag = 2
            Sol = exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers)
            Count_select = UnSelected.copy()
            UnSelected[0] += 1;  UnSelected[1] = 1
            F3_Explore += PF[2];  F3_Exploit = PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Exploit[2], Seq_Cost_Exploit[1] = Seq_Cost_Exploit[1], Seq_Cost_Exploit[0]
            Seq_Cost_Exploit[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Exploit[0]:
                PF_F3.append(Seq_Cost_Exploit[0])

        if TBest.Cost < Best.Cost:
            Best = TBest

        if Flag_Change != SelectFlag:
            Flag_Change = SelectFlag
            Seq_Time_Explore[2], Seq_Time_Explore[1], Seq_Time_Explore[0] = \
                Seq_Time_Explore[1], Seq_Time_Explore[0], Count_select[0]
            Seq_Time_Exploit[2], Seq_Time_Exploit[1], Seq_Time_Exploit[0] = \
                Seq_Time_Exploit[1], Seq_Time_Exploit[0], Count_select[1]

        if Score_Explore < Score_Exploit:
            Mega_Explor  = max(Mega_Explor  - 0.01, 0.01)
            Mega_Exploit = 0.99
        else:
            Mega_Explor  = 0.99
            Mega_Exploit = max(Mega_Exploit - 0.01, 0.01)

        print_iteration_log(Iter, Sol, Best, time.perf_counter() - t_start)

        if Best.metrics['hc_count'] == 0:
            print(f"[INFO] Zero HC violations achieved at iteration {Iter}.")
            break

        # ── Stagnation recovery ───────────────────────────────────────────────
        if Best.Cost >= prev_best:
            stagnation += 1
        else:
            stagnation = 0
            prev_best  = Best.Cost

        if stagnation >= 10:
            print(f"[WARN] Stagnation at iter {Iter} — injecting diversity...")
            n_replace = nSol // 3
            new_vecs  = [lb + (ub - lb) * np.random.rand(dim) for _ in range(n_replace)]
            res       = parallel_eval(new_vecs, n_workers)
            for k, (c, a, met) in enumerate(res):
                Sol[-(k + 1)] = Solution(new_vecs[k], c, a, met)
            stagnation = 0

    print(f"\n[INFO] Optimasi selesai dalam {time.perf_counter() - t_start:.1f}s")
    return Best


# ==========================================
# 7. Eksekusi & Ekspor
# ==========================================
if __name__ == '__main__':
    nSol    = 100
    MaxIter = 100
    dim     = _N_COURSES
    lb, ub  = 0.0, 1.0

    t0 = time.perf_counter()
    best_solution = run_puma_optimizer(nSol, MaxIter, lb, ub, dim, n_workers=None)
    print(f"Total elapsed: {time.perf_counter() - t0:.1f}s")

    res_list = []
    for a in best_solution.assignments:
        course = mk_df_raw.iloc[a['course_i']]
        room   = rooms_df.iloc[a['room']]
        t1     = timeslots_df.iloc[a['start_slot']]
        t2     = timeslots_df.iloc[a['end_slot']]
        res_list.append({
            'event_id':    course['event_id'],
            'course_name': course['course_name'],
            'teacher_id':  course['teacher_id'],
            'room_id':     room['room_id'],
            'day':         t1['day'],
            'time_start':  t1['time_start'],
            'time_end':    t2['time_end'],
        })

    res_df   = pd.DataFrame(res_list)
    out_path = 'scheduled_courses_poa_v5.csv'
    res_df.to_csv(out_path, index=False)

    m = best_solution.metrics
    print(f"\n{'=' * 65}")
    print(f"  FINAL RESULT")
    print(f"  HC Violations  : {m['hc_count']}  (penalty: {m['hc_value']:,})")
    print(f"  Repaired by P2 : {m.get('repaired', 0)}")
    print(f"  SC Violations  : {m['sc_count']}  (penalty: {m['sc_value']})")
    print(f"  Total Fitness  : {best_solution.Cost:,}")
    print(f"  Courses scheduled: {len(res_list)} / {_N_COURSES}")
    print(f"{'=' * 65}")
    print(f"[INFO] Solusi disimpan di '{out_path}'")

### Repaired Mechanism with Logs

In [2]:
import numpy as np
import pandas as pd
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
import os

# ==========================================
# 1. Deklarasi & Persiapan Dataset
# ==========================================
print("[INFO] Memuat dan memproses dataset...")
mk_df_raw = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/mk_dsi_genap_praktikum_processed.csv')
rooms_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/rooms_processed.csv')
timeslots_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/timeslots_processed.csv')

# ── Pre-compute valid consecutive timeslot pairs ──────────────────────────────
valid_starts = [
    i for i in range(len(timeslots_df) - 1)
    if (timeslots_df.iloc[i]['day'] == timeslots_df.iloc[i + 1]['day'] and
        timeslots_df.iloc[i]['time_end'] == timeslots_df.iloc[i + 1]['time_start'])
]

rooms_prac = rooms_df[rooms_df['is_practicum'] == True].index.tolist()
rooms_theo = rooms_df[rooms_df['is_theory'] == True].index.tolist()

# ── Teacher workload heuristic ────────────────────────────────────────────────
teacher_workload = mk_df_raw['teacher_id'].value_counts().to_dict()
mk_df_raw['teacher_degree'] = mk_df_raw['teacher_id'].map(teacher_workload)

# ── Pre-build global slot lists (computed ONCE) ───────────────────────────────
PRAC_SLOTS = [(r, s) for r in rooms_prac for s in valid_starts]
THEO_SLOTS = [(r, s) for r in rooms_theo for s in valid_starts]

# ── Pre-extracted numpy arrays for O(1) hot-loop access ──────────────────────
_REQ_PRAC   = mk_df_raw['requires_practicum_room'].to_numpy()
_TEACHER    = mk_df_raw['teacher_id'].to_numpy()
_T_PRIORITY = mk_df_raw['teacher_priority'].to_numpy()
_T_DEGREE   = mk_df_raw['teacher_degree'].to_numpy()
_TS_DAY     = timeslots_df['day'].to_numpy()
_TS_SESSION = timeslots_df['session_type'].to_numpy()
_N_COURSES  = len(mk_df_raw)

SESSION_MORNING   = (_TS_SESSION == 'Morning')
SESSION_AFTERNOON = (_TS_SESSION == 'Afternoon')

# ── HC weight: any 1 HC violation > any possible SC total ────────────────────
HC_WEIGHT = 10_000_000

# ── Global flag: controls whether repair logs are printed (disabled in workers) ──
_VERBOSE_REPAIR = False


# ==========================================
# 2. Fitness Function with Conflict-Directed Repair
# ==========================================
def fitness_function(sol_x, verbose_repair=False):
    """
    Three-phase decoder with detailed repair logging.

    PHASE 1 — Greedy HC-aware assignment
    PHASE 2 — Conflict-Directed Repair (with per-course trace when verbose_repair=True)
    PHASE 3 — SC Re-evaluation
    """
    hc_count     = 0
    sc_value     = 0
    sc_count     = 0
    assignments      = []
    unassigned_idxs  = []

    room_time_booked    = set()
    teacher_time_booked = set()
    teacher_day_rooms   = {}
    teacher_ts_to_asgn  = {}

    avail_prac = set(range(len(PRAC_SLOTS)))
    avail_theo = set(range(len(THEO_SLOTS)))

    # Repair trace: list of dicts, one per unassigned course processed
    repair_log = []

    # ── MCF + PO random-key ordering ─────────────────────────────────────────
    base_scores = (_REQ_PRAC.astype(float) * 1000) + (_T_DEGREE * 10)
    priorities  = base_scores + (sol_x * 100)
    queue_order = np.argsort(-priorities)

    # =========================================================================
    # PHASE 1 — GREEDY ASSIGNMENT
    # =========================================================================
    for idx in queue_order:
        req_prac   = bool(_REQ_PRAC[idx])
        teacher    = _TEACHER[idx]
        t_priority = int(_T_PRIORITY[idx])

        slot_list = PRAC_SLOTS if req_prac else THEO_SLOTS
        avail_set = avail_prac  if req_prac else avail_theo

        hc_fallback    = None
        best_sc        = float('inf')
        best_slot_info = None
        sc_scan_count  = 0

        for si in list(avail_set):
            r, s = slot_list[si]
            t1, t2 = s, s + 1

            if (teacher, t1) in teacher_time_booked or (teacher, t2) in teacher_time_booked:
                continue

            if hc_fallback is None:
                hc_fallback = (r, s, _TS_DAY[s], t1, t2, si)

            day     = _TS_DAY[s]
            temp_sc = 0
            if t_priority == 1 and not (SESSION_MORNING[t1] and SESSION_MORNING[t2]):
                temp_sc += 10
            elif t_priority == 2 and not (SESSION_AFTERNOON[t1] and SESSION_AFTERNOON[t2]):
                temp_sc += 10
            if teacher in teacher_day_rooms and day in teacher_day_rooms[teacher]:
                if r not in teacher_day_rooms[teacher][day]:
                    temp_sc += 50

            if temp_sc < best_sc:
                best_sc        = temp_sc
                best_slot_info = (r, s, day, t1, t2, si)
                if temp_sc == 0:
                    break
            sc_scan_count += 1
            if sc_scan_count >= 50:
                break

        chosen = best_slot_info if best_slot_info is not None else hc_fallback

        if chosen is not None:
            r, s, day, t1, t2, si = chosen
            _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
                       assignments, room_time_booked, teacher_time_booked,
                       teacher_day_rooms, teacher_ts_to_asgn,
                       avail_prac, avail_theo)
        else:
            unassigned_idxs.append(int(idx))

    # =========================================================================
    # PHASE 2 — CONFLICT-DIRECTED REPAIR
    # =========================================================================
    still_unassigned = []

    for idx in unassigned_idxs:
        req_prac   = bool(_REQ_PRAC[idx])
        teacher    = _TEACHER[idx]
        t_priority = int(_T_PRIORITY[idx])
        course_name = mk_df_raw['course_name'].iloc[idx] if verbose_repair else None

        slot_list = PRAC_SLOTS if req_prac else THEO_SLOTS
        avail_set = avail_prac  if req_prac else avail_theo

        placed           = False
        attempts         = 0
        failed_relocations = 0
        trace_steps      = []   # repair steps for this course

        for r, s in slot_list:
            t1, t2 = s, s + 1

            if (r, t1) in room_time_booked or (r, t2) in room_time_booked:
                continue

            blocked_t1 = (teacher, t1) in teacher_time_booked
            blocked_t2 = (teacher, t2) in teacher_time_booked

            # ── Slot is unexpectedly free → direct assign ─────────────────────
            if not blocked_t1 and not blocked_t2:
                day = _TS_DAY[s]
                _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
                           assignments, room_time_booked, teacher_time_booked,
                           teacher_day_rooms, teacher_ts_to_asgn,
                           avail_prac, avail_theo)
                placed = True
                if verbose_repair:
                    trace_steps.append({
                        'action': 'DIRECT',
                        'detail': f"Room {rooms_df.iloc[r]['room_id']} | "
                                  f"{_TS_DAY[s]} {timeslots_df.iloc[s]['time_start']}–"
                                  f"{timeslots_df.iloc[t2]['time_end']} (no blocker)"
                    })
                break

            # ── Find blocker ──────────────────────────────────────────────────
            blocker_key      = (teacher, t1) if blocked_t1 else (teacher, t2)
            blocker_asgn_idx = teacher_ts_to_asgn.get(blocker_key)
            if blocker_asgn_idx is None:
                continue

            blocker    = assignments[blocker_asgn_idx]
            b_idx      = blocker['course_i']
            b_req_prac = bool(_REQ_PRAC[b_idx])
            b_teacher  = blocker['teacher']
            b_t1_old   = blocker['start_slot']
            b_t2_old   = blocker['end_slot']
            b_r_old    = blocker['room']
            b_priority = blocker['priority']
            b_day_old  = blocker['day']
            b_course_name = mk_df_raw['course_name'].iloc[b_idx] if verbose_repair else None

            attempts += 1

            # ── Temporarily unbook blocker ────────────────────────────────────
            room_time_booked.discard((b_r_old, b_t1_old))
            room_time_booked.discard((b_r_old, b_t2_old))
            teacher_time_booked.discard((b_teacher, b_t1_old))
            teacher_time_booked.discard((b_teacher, b_t2_old))
            teacher_ts_to_asgn.pop((b_teacher, b_t1_old), None)
            teacher_ts_to_asgn.pop((b_teacher, b_t2_old), None)

            if b_teacher in teacher_day_rooms and b_day_old in teacher_day_rooms[b_teacher]:
                try:
                    teacher_day_rooms[b_teacher][b_day_old].remove(b_r_old)
                except ValueError:
                    pass
                if not teacher_day_rooms[b_teacher][b_day_old]:
                    del teacher_day_rooms[b_teacher][b_day_old]

            b_slot_list = PRAC_SLOTS if b_req_prac else THEO_SLOTS
            b_avail_set = avail_prac  if b_req_prac else avail_theo
            b_si = -1
            try:
                b_si = b_slot_list.index((b_r_old, b_t1_old))
                b_avail_set.add(b_si)
            except ValueError:
                pass

            # ── Search for blocker relocation ─────────────────────────────────
            reloc_found = None
            b_avail     = avail_prac if b_req_prac else avail_theo
            b_slots     = PRAC_SLOTS if b_req_prac else THEO_SLOTS
            for si2 in list(b_avail):
                r2, s2 = b_slots[si2]
                t1n, t2n = s2, s2 + 1
                if (r2, t1n) in room_time_booked or (r2, t2n) in room_time_booked:
                    continue
                if (b_teacher, t1n) in teacher_time_booked or (b_teacher, t2n) in teacher_time_booked:
                    continue
                reloc_found = (r2, s2, _TS_DAY[s2], t1n, t2n, si2)
                break

            if reloc_found is not None:
                r2, s2, day2, t1n, t2n, si2 = reloc_found

                _do_assign(b_idx, r2, s2, day2, t1n, t2n, b_priority, b_teacher,
                           b_req_prac, assignments, room_time_booked,
                           teacher_time_booked, teacher_day_rooms,
                           teacher_ts_to_asgn, avail_prac, avail_theo,
                           update_idx=blocker_asgn_idx)

                day = _TS_DAY[s]
                _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
                           assignments, room_time_booked, teacher_time_booked,
                           teacher_day_rooms, teacher_ts_to_asgn,
                           avail_prac, avail_theo)
                placed = True

                if verbose_repair:
                    trace_steps.append({
                        'action': 'SWAP_SUCCESS',
                        'detail': (
                            f"Blocker [{b_course_name} | T:{b_teacher}] moved "
                            f"{rooms_df.iloc[b_r_old]['room_id']} "
                            f"{_TS_DAY[b_t1_old]} {timeslots_df.iloc[b_t1_old]['time_start']}–"
                            f"{timeslots_df.iloc[b_t2_old]['time_end']}"
                            f"  →  "
                            f"{rooms_df.iloc[r2]['room_id']} "
                            f"{day2} {timeslots_df.iloc[s2]['time_start']}–"
                            f"{timeslots_df.iloc[t2n]['time_end']}"
                        )
                    })
                break

            else:
                failed_relocations += 1
                if verbose_repair:
                    trace_steps.append({
                        'action': 'SWAP_FAILED',
                        'detail': (
                            f"Blocker [{b_course_name} | T:{b_teacher}] at "
                            f"{rooms_df.iloc[b_r_old]['room_id']} "
                            f"{_TS_DAY[b_t1_old]} {timeslots_df.iloc[b_t1_old]['time_start']} "
                            f"has no free alternative slot"
                        )
                    })

                # Restore blocker
                room_time_booked.add((b_r_old, b_t1_old))
                room_time_booked.add((b_r_old, b_t2_old))
                teacher_time_booked.add((b_teacher, b_t1_old))
                teacher_time_booked.add((b_teacher, b_t2_old))
                teacher_ts_to_asgn[(b_teacher, b_t1_old)] = blocker_asgn_idx
                teacher_ts_to_asgn[(b_teacher, b_t2_old)] = blocker_asgn_idx
                if b_teacher not in teacher_day_rooms:
                    teacher_day_rooms[b_teacher] = {}
                if b_day_old not in teacher_day_rooms[b_teacher]:
                    teacher_day_rooms[b_teacher][b_day_old] = []
                teacher_day_rooms[b_teacher][b_day_old].append(b_r_old)
                if b_si >= 0:
                    b_avail_set.discard(b_si)

        # Record repair outcome for this course
        if verbose_repair:
            repair_log.append({
                'course_i':    idx,
                'course_name': course_name,
                'teacher':     teacher,
                'type':        'Practicum' if req_prac else 'Theory',
                'placed':      placed,
                'attempts':    attempts,
                'failed_relocs': failed_relocations,
                'steps':       trace_steps,
            })

        if not placed:
            still_unassigned.append(idx)

    hc_count = len(still_unassigned)

    # =========================================================================
    # PHASE 3 — SC TOTAL EVALUATION
    # =========================================================================
    for asgn in assignments:
        t1 = asgn['start_slot']
        t2 = asgn['end_slot']
        tp = asgn['priority']
        if tp == 1 and not (SESSION_MORNING[t1] and SESSION_MORNING[t2]):
            sc_value += 10
        elif tp == 2 and not (SESSION_AFTERNOON[t1] and SESSION_AFTERNOON[t2]):
            sc_value += 10

    for teacher_id, days in teacher_day_rooms.items():
        for day, rooms in days.items():
            unique_rooms = len(set(rooms))
            if unique_rooms > 1:
                v = unique_rooms - 1
                sc_count += v
                sc_value += 50 * v

    fitness = hc_count * HC_WEIGHT + sc_value
    metrics = {
        'hc_count':   hc_count,
        'hc_value':   hc_count * HC_WEIGHT,
        'sc_count':   sc_count,
        'sc_value':   sc_value,
        'repaired':   len(unassigned_idxs) - hc_count,
        'unassigned': len(unassigned_idxs),
        'repair_log': repair_log,
    }
    return fitness, assignments, metrics


# ==========================================
# 3. Assignment Helper
# ==========================================
def _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
               assignments, room_time_booked, teacher_time_booked,
               teacher_day_rooms, teacher_ts_to_asgn,
               avail_prac, avail_theo, update_idx=None):
    room_time_booked.add((r, t1)); room_time_booked.add((r, t2))
    teacher_time_booked.add((teacher, t1)); teacher_time_booked.add((teacher, t2))

    slot_list = PRAC_SLOTS if req_prac else THEO_SLOTS
    avail_set = avail_prac  if req_prac else avail_theo
    for si2 in list(avail_set):
        r2, s2 = slot_list[si2]
        if r2 == r and (s2 == t1 or s2 == t2 or s2 + 1 == t1):
            avail_set.discard(si2)

    if teacher not in teacher_day_rooms:
        teacher_day_rooms[teacher] = {}
    if day not in teacher_day_rooms[teacher]:
        teacher_day_rooms[teacher][day] = []
    teacher_day_rooms[teacher][day].append(r)

    asgn = {
        'course_i': int(idx), 'room': r,
        'start_slot': s, 'end_slot': t2,
        'teacher': teacher, 'priority': t_priority, 'day': day
    }

    if update_idx is not None:
        assignments[update_idx] = asgn
        teacher_ts_to_asgn[(teacher, t1)] = update_idx
        teacher_ts_to_asgn[(teacher, t2)] = update_idx
    else:
        new_idx = len(assignments)
        assignments.append(asgn)
        teacher_ts_to_asgn[(teacher, t1)] = new_idx
        teacher_ts_to_asgn[(teacher, t2)] = new_idx


def _eval_worker(x):
    return fitness_function(x, verbose_repair=False)


# ==========================================
# 4. Solution Class
# ==========================================
class Solution:
    __slots__ = ('X', 'Cost', 'assignments', 'metrics')

    def __init__(self, x, cost, assignments=None, metrics=None):
        self.X           = x
        self.Cost        = cost
        self.assignments = assignments
        self.metrics     = metrics

    def clone(self):
        return Solution(self.X.copy(), self.Cost, self.assignments, self.metrics)


# ==========================================
# 5. Parallel Population Evaluator
# ==========================================
def parallel_eval(vectors, n_workers=None):
    n_workers = n_workers or min(os.cpu_count(), len(vectors))
    if n_workers <= 1 or len(vectors) <= 4:
        return [fitness_function(v) for v in vectors]
    results = [None] * len(vectors)
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_eval_worker, v): i for i, v in enumerate(vectors)}
        for fut in as_completed(futures):
            results[futures[fut]] = fut.result()
    return results


# ==========================================
# 6. Logging Helpers
# ==========================================
_LOG_SEP  = "─" * 110
_LOG_HDR  = "═" * 110

def print_iteration_log(Iter, Sol, Best, elapsed, phase_label=""):
    """Full per-iteration summary line."""
    worst_cost = max(s.Cost for s in Sol)
    m = Best.metrics
    tag = f"[{phase_label}] " if phase_label else ""
    print(
        f"{tag}Iter {Iter:03d} | "
        f"Best Fitness: {Best.Cost:>14,.0f} | Worst Fitness: {worst_cost:>14,.0f} | "
        f"HC Viol: {m['hc_count']:>3d} (Penalty: {m['hc_value']:>14,.0f}) | "
        f"SC Viol: {m['sc_count']:>4d} (Penalty: {m['sc_value']:>7,.0f}) | "
        f"Time: {elapsed:>7.1f}s"
    )


def print_repair_detail(metrics, iter_label="Best Solution"):
    """Print the per-course repair trace stored in metrics['repair_log']."""
    log = metrics.get('repair_log', [])
    if not log:
        return

    n_unassigned = metrics['unassigned']
    n_repaired   = metrics['repaired']
    n_failed     = metrics['hc_count']

    print()
    print(_LOG_HDR)
    print(f"  REPAIR DETAIL — {iter_label}".center(110))
    print(f"  Courses sent to Phase 2: {n_unassigned}  |  "
          f"Successfully repaired: {n_repaired}  |  "
          f"Still unassigned (HC): {n_failed}".center(110))
    print(_LOG_HDR)

    for i, entry in enumerate(log, 1):
        status = "✔ PLACED" if entry['placed'] else "✘ FAILED"
        room_type = entry['type']
        print(f"\n  [{i:02d}/{len(log)}] {status} | "
              f"Course : {entry['course_name']}  "
              f"| Teacher: {entry['teacher']}  "
              f"| Type: {room_type}")
        print(f"         Swap attempts: {entry['attempts']}  |  "
              f"Failed relocations: {entry['failed_relocs']}")

        if entry['steps']:
            for step in entry['steps']:
                icon = "  ↳ " if step['action'] != 'SWAP_FAILED' else "  ✗ "
                print(f"    {icon}[{step['action']:>14s}] {step['detail']}")
        else:
            print("    (no blocker swap required — assigned directly)")

    print(_LOG_SEP)


def print_header(title):
    print("\n" + _LOG_HDR)
    print(f"  {title}".center(110))
    print(_LOG_HDR)
    print(
        f"{'Iter':>6} | "
        f"{'Best Fitness':>14} | {'Worst Fitness':>14} | "
        f"{'HC Viol':>7} | {'HC Penalty':>14} | "
        f"{'SC Viol':>7} | {'SC Penalty':>10} | "
        f"{'Time':>8}"
    )
    print(_LOG_SEP)


def print_summary_line(Iter, Sol, Best, elapsed):
    """Compact tabular line (used inside the header table)."""
    worst_cost = max(s.Cost for s in Sol)
    m = Best.metrics
    print(
        f"{Iter:>6} | "
        f"{Best.Cost:>14,.0f} | {worst_cost:>14,.0f} | "
        f"{m['hc_count']:>7d} | {m['hc_value']:>14,.0f} | "
        f"{m['sc_count']:>7d} | {m['sc_value']:>10,.0f} | "
        f"{elapsed:>7.1f}s"
    )


# ==========================================
# 7. Puma Optimizer — MECHANISM UNCHANGED
# ==========================================
def exploration(Sol, lb, ub, dim, nSol, n_workers):
    pCR = 0.20
    PCR = 1 - pCR
    p   = PCR / nSol
    Sol.sort(key=lambda s: s.Cost)

    new_vectors = []
    for i in range(nSol):
        x = Sol[i].X
        A = list(range(nSol)); A.remove(i); np.random.shuffle(A)
        a, b, c, d, e, f = A[:6]
        G = 2 * np.random.rand() - 1
        if np.random.rand() < 0.5:
            y = np.random.rand(dim) * (ub - lb) + lb
        else:
            da = Sol[a].X - Sol[b].X
            db = Sol[c].X - Sol[d].X
            dc = Sol[e].X - Sol[f].X
            y  = Sol[a].X + G * da + G * ((da - db) + (db - dc))
        y  = np.clip(y, lb, ub)
        z  = x.copy()
        j0 = np.random.randint(dim)
        mask = (np.random.rand(dim) <= pCR)
        mask[j0] = True
        z[mask] = y[mask]
        new_vectors.append(z)

    results = parallel_eval(new_vectors, n_workers)
    for i, (cost, asg, met) in enumerate(results):
        candidate = Solution(new_vectors[i], cost, asg, met)
        if candidate.Cost < Sol[i].Cost:
            Sol[i] = candidate
        else:
            pCR = pCR + p
    return Sol


def exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers):
    Q, Beta = 0.67, 2
    mbest   = np.mean([s.X for s in Sol], axis=0)

    new_vectors = []
    for i in range(nSol):
        beta1 = 2 * np.random.rand()
        beta2 = np.random.randn(dim)
        w, v  = np.random.randn(dim), np.random.randn(dim)
        F1    = np.random.randn(dim) * np.exp(2 - Iter * (2 / MaxIter))
        F2    = w * (v ** 2) * np.cos(2 * np.random.rand() * w)
        R_1   = 2 * np.random.rand() - 1
        S1    = (2 * np.random.rand() - 1 + np.random.randn(dim))
        S2    = F1 * R_1 * Sol[i].X + F2 * (1 - R_1) * Best.X
        VEC   = S2 / (S1 + 1e-10)

        if np.random.rand() <= 0.5:
            if np.random.rand() > Q:
                r_idx = np.random.randint(nSol)
                x_new = Best.X + beta1 * np.exp(beta2) * (Sol[r_idx].X - Sol[i].X)
            else:
                x_new = beta1 * VEC - Best.X
        else:
            r1    = np.random.randint(nSol)
            sign  = (-1) ** np.random.randint(0, 2)
            x_new = (mbest * Sol[r1].X - sign * Sol[i].X) / (1 + Beta * np.random.rand())

        new_vectors.append(np.clip(x_new, lb, ub))

    results = parallel_eval(new_vectors, n_workers)
    for i, (cost, asg, met) in enumerate(results):
        candidate = Solution(new_vectors[i], cost, asg, met)
        if candidate.Cost < Sol[i].Cost:
            Sol[i] = candidate
    return Sol


def run_puma_optimizer(nSol, MaxIter, lb, ub, dim, n_workers=None):
    n_workers = n_workers or min(os.cpu_count(), nSol)

    print(_LOG_HDR)
    print("  PUMA OPTIMIZER v5 — CONFIGURATION".center(110))
    print(_LOG_HDR)
    print(f"  Parallel workers  : {n_workers}")
    print(f"  Population (nSol) : {nSol}")
    print(f"  Max iterations    : {MaxIter}")
    print(f"  Dimensions (courses): {_N_COURSES}")
    print(f"  HC weight         : {HC_WEIGHT:,}")
    print(f"  Rooms (theory)    : {len(rooms_theo)}  |  Rooms (practicum): {len(rooms_prac)}")
    print(f"  Valid timeslot pairs: {len(valid_starts)}")
    print(_LOG_SEP)

    # ── Initial population ────────────────────────────────────────────────────
    print("\n[INIT] Evaluating initial population...")
    init_vectors    = [lb + (ub - lb) * np.random.rand(dim) for _ in range(nSol)]
    init_vectors[0] = np.full(dim, 0.5)

    results      = parallel_eval(init_vectors, n_workers)
    Sol          = [Solution(init_vectors[i], *results[i]) for i in range(nSol)]
    Best         = min(Sol, key=lambda s: s.Cost)
    Initial_Best = Best.clone()

    m = Best.metrics
    print(f"\n[INIT] Initial Best Solution:")
    print(f"       Fitness     : {Best.Cost:,.0f}")
    print(f"       HC Violations: {m['hc_count']}  (Penalty: {m['hc_value']:,})")
    print(f"       SC Violations: {m['sc_count']}  (Penalty: {m['sc_value']:,})")
    print(f"       Courses sent to repair: {m['unassigned']}  |  Repaired: {m['repaired']}")

    # Show repair detail for initial best
    if m['unassigned'] > 0:
        # Re-evaluate with verbose to get repair trace
        _, _, m_verbose = fitness_function(Best.X, verbose_repair=True)
        print_repair_detail(m_verbose, iter_label="INITIAL POPULATION")

    if Best.metrics['hc_count'] == 0:
        print("\n[INFO] Zero HC violations at initialisation — skipping optimisation.")
        return Best

    # ── PO state variables (UNCHANGED) ───────────────────────────────────────
    UnSelected = [1, 1]
    F3_Explore = F3_Exploit = 0
    Seq_Time_Explore  = [1, 1, 1];  Seq_Time_Exploit  = [1, 1, 1]
    Seq_Cost_Explore  = [1, 1, 1];  Seq_Cost_Exploit  = [1, 1, 1]
    PF            = [0.5, 0.5, 0.3]
    PF_F3         = []
    Mega_Explor   = Mega_Exploit = 0.99
    Costs_Explor  = np.zeros(3)
    Costs_Exploit = np.zeros(3)
    Flag_Change   = 1

    print_header("PUMA OPTIMISATION PROGRESS")
    t_start      = time.perf_counter()
    prev_best_hc = Best.metrics['hc_count']  # track HC drops for repair-detail triggers

    # ── Bootstrap iterations 1–3 ──────────────────────────────────────────────
    for Iter in range(1, 4):
        Sol_E1 = exploration ([s.clone() for s in Sol], lb, ub, dim, nSol, n_workers)
        Sol_E2 = exploitation([s.clone() for s in Sol], lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers)
        Costs_Explor[Iter - 1]  = min(s.Cost for s in Sol_E1)
        Costs_Exploit[Iter - 1] = min(s.Cost for s in Sol_E2)

        merged = Sol + Sol_E1 + Sol_E2
        merged.sort(key=lambda s: s.Cost)
        Sol  = merged[:nSol]
        Best = Sol[0]
        elapsed = time.perf_counter() - t_start
        print_summary_line(Iter, Sol, Best, elapsed)

        # Show repair detail whenever HC count drops
        if Best.metrics['hc_count'] < prev_best_hc and Best.metrics['unassigned'] > 0:
            _, _, m_v = fitness_function(Best.X, verbose_repair=True)
            print_repair_detail(m_v, iter_label=f"Iter {Iter:03d} (HC improved: "
                                                 f"{prev_best_hc} → {Best.metrics['hc_count']})")
            prev_best_hc = Best.metrics['hc_count']

        if Best.metrics['hc_count'] == 0:
            print(_LOG_SEP)
            print(f"[INFO] Zero HC achieved at bootstrap iteration {Iter}.")
            return Best

    Seq_Cost_Explore[0] = abs(Initial_Best.Cost  - Costs_Explor[0])
    Seq_Cost_Explore[1] = abs(Costs_Explor[1]    - Costs_Explor[0])
    Seq_Cost_Explore[2] = abs(Costs_Explor[2]    - Costs_Explor[1])
    Seq_Cost_Exploit[0] = abs(Initial_Best.Cost  - Costs_Exploit[0])
    Seq_Cost_Exploit[1] = abs(Costs_Exploit[1]   - Costs_Exploit[0])
    Seq_Cost_Exploit[2] = abs(Costs_Exploit[2]   - Costs_Exploit[1])

    for v in Seq_Cost_Explore + Seq_Cost_Exploit:
        if v != 0:
            PF_F3.append(v)

    # ── Main loop (PO mechanism UNCHANGED) ───────────────────────────────────
    stagnation = 0
    prev_best  = Best.Cost

    for Iter in range(4, MaxIter + 1):
        F1_Explor  = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0])
        F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
        F2_Explor  = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore))
        F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
        Score_Explore  = Mega_Explor  * (F1_Explor  + F2_Explor)
        Score_Exploit  = Mega_Exploit * (F1_Exploit + F2_Exploit)

        if Score_Explore > Score_Exploit:
            SelectFlag = 1
            Sol = exploration(Sol, lb, ub, dim, nSol, n_workers)
            Count_select = UnSelected.copy()
            UnSelected[1] += 1;  UnSelected[0] = 1
            F3_Explore = PF[2];  F3_Exploit += PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Explore[2], Seq_Cost_Explore[1] = Seq_Cost_Explore[1], Seq_Cost_Explore[0]
            Seq_Cost_Explore[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Explore[0]:
                PF_F3.append(Seq_Cost_Explore[0])
        else:
            SelectFlag = 2
            Sol = exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers)
            Count_select = UnSelected.copy()
            UnSelected[0] += 1;  UnSelected[1] = 1
            F3_Explore += PF[2];  F3_Exploit = PF[2]
            TBest = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Exploit[2], Seq_Cost_Exploit[1] = Seq_Cost_Exploit[1], Seq_Cost_Exploit[0]
            Seq_Cost_Exploit[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Exploit[0]:
                PF_F3.append(Seq_Cost_Exploit[0])

        if TBest.Cost < Best.Cost:
            Best = TBest

        if Flag_Change != SelectFlag:
            Flag_Change = SelectFlag
            Seq_Time_Explore[2], Seq_Time_Explore[1], Seq_Time_Explore[0] = \
                Seq_Time_Explore[1], Seq_Time_Explore[0], Count_select[0]
            Seq_Time_Exploit[2], Seq_Time_Exploit[1], Seq_Time_Exploit[0] = \
                Seq_Time_Exploit[1], Seq_Time_Exploit[0], Count_select[1]

        if Score_Explore < Score_Exploit:
            Mega_Explor  = max(Mega_Explor  - 0.01, 0.01)
            Mega_Exploit = 0.99
        else:
            Mega_Explor  = 0.99
            Mega_Exploit = max(Mega_Exploit - 0.01, 0.01)

        elapsed = time.perf_counter() - t_start
        print_summary_line(Iter, Sol, Best, elapsed)

        # Show repair detail on every HC improvement
        if Best.metrics['hc_count'] < prev_best_hc and Best.metrics['unassigned'] > 0:
            _, _, m_v = fitness_function(Best.X, verbose_repair=True)
            print_repair_detail(m_v, iter_label=f"Iter {Iter:03d} (HC improved: "
                                                 f"{prev_best_hc} → {Best.metrics['hc_count']})")
            prev_best_hc = Best.metrics['hc_count']

        if Best.metrics['hc_count'] == 0:
            print(_LOG_SEP)
            print(f"[INFO] Zero HC violations achieved at iteration {Iter}.")
            break

        # ── Stagnation recovery ───────────────────────────────────────────────
        if Best.Cost >= prev_best:
            stagnation += 1
        else:
            stagnation = 0
            prev_best  = Best.Cost

        if stagnation >= 10:
            print(_LOG_SEP)
            print(f"[WARN] Stagnation at iter {Iter} — injecting diversity into bottom third...")
            n_replace = nSol // 3
            new_vecs  = [lb + (ub - lb) * np.random.rand(dim) for _ in range(n_replace)]
            res       = parallel_eval(new_vecs, n_workers)
            for k, (c, a, met) in enumerate(res):
                Sol[-(k + 1)] = Solution(new_vecs[k], c, a, met)
            stagnation = 0
            print(_LOG_SEP)

    total_time = time.perf_counter() - t_start
    print(f"\n[INFO] Optimisation finished in {total_time:.1f}s")
    return Best


# ==========================================
# 8. Eksekusi & Ekspor
# ==========================================
if __name__ == '__main__':
    nSol    = 300
    MaxIter = 500
    dim     = _N_COURSES
    lb, ub  = 0.0, 1.0

    t0            = time.perf_counter()
    best_solution = run_puma_optimizer(nSol, MaxIter, lb, ub, dim, n_workers=None)
    total_elapsed = time.perf_counter() - t0

    # ── Final repair detail ───────────────────────────────────────────────────
    if best_solution.metrics['unassigned'] > 0:
        _, _, m_final = fitness_function(best_solution.X, verbose_repair=True)
        print_repair_detail(m_final, iter_label="FINAL BEST SOLUTION")

    # ── Build result DataFrame ────────────────────────────────────────────────
    res_list = []
    for a in best_solution.assignments:
        course = mk_df_raw.iloc[a['course_i']]
        room   = rooms_df.iloc[a['room']]
        t1     = timeslots_df.iloc[a['start_slot']]
        t2     = timeslots_df.iloc[a['end_slot']]
        res_list.append({
            'event_id':    course['event_id'],
            'course_name': course['course_name'],
            'teacher_id':  course['teacher_id'],
            'room_id':     room['room_id'],
            'day':         t1['day'],
            'time_start':  t1['time_start'],
            'time_end':    t2['time_end'],
        })

    res_df   = pd.DataFrame(res_list)
    out_path = 'scheduled_courses_poa_v6.csv'
    res_df.to_csv(out_path, index=False)

    m = best_solution.metrics
    print("\n" + _LOG_HDR)
    print("  FINAL RESULT SUMMARY".center(110))
    print(_LOG_HDR)
    print(f"  Total elapsed         : {total_elapsed:.1f}s")
    print(f"  Courses scheduled     : {len(res_list)} / {_N_COURSES}")
    print(f"  HC Violations         : {m['hc_count']}   (Penalty: {m['hc_value']:,})")
    print(f"  SC Violations         : {m['sc_count']}   (Penalty: {m['sc_value']:,})")
    print(f"  Total Fitness Score   : {best_solution.Cost:,}")
    print(f"  Courses repaired (P2) : {m['repaired']} / {m['unassigned']}")
    print(f"  Output saved to       : '{out_path}'")
    print(_LOG_HDR)

[INFO] Memuat dan memproses dataset...
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
                                       PUMA OPTIMIZER v5 — CONFIGURATION                                      
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Parallel workers  : 8
  Population (nSol) : 300
  Max iterations    : 500
  Dimensions (courses): 725
  HC weight         : 10,000,000
  Rooms (theory)    : 39  |  Rooms (practicum): 12
  Valid timeslot pairs: 55
──────────────────────────────────────────────────────────────────────────────────────────────────────────────

[INIT] Evaluating initial population...

[INIT] Initial Best Solution:
       Fitness     : 5,370
       HC Violations: 0  (Penalty: 0)
       SC Violations: 55  (Penalty: 5,370)
       Courses sent to repair: 40  |  Repaired: 40

══════════════════════════════════════════════════════════════════════

In [3]:
import numpy as np
import pandas as pd
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
import os

# ==========================================
# 1. Deklarasi & Persiapan Dataset
# ==========================================
print("[INFO] Memuat dan memproses dataset...")
mk_df_raw    = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/mk_dsi_genap_praktikum_processed.csv')
rooms_df     = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/rooms_processed.csv')
timeslots_df = pd.read_csv('/home/emery/Documents/Scheduling_Puma_Optimizer/app/assets/timeslots_processed.csv')

# ── Pre-compute valid consecutive timeslot pairs ──────────────────────────────
valid_starts = [
    i for i in range(len(timeslots_df) - 1)
    if (timeslots_df.iloc[i]['day'] == timeslots_df.iloc[i + 1]['day'] and
        timeslots_df.iloc[i]['time_end'] == timeslots_df.iloc[i + 1]['time_start'])
]

rooms_prac = rooms_df[rooms_df['is_practicum'] == True].index.tolist()
rooms_theo = rooms_df[rooms_df['is_theory'] == True].index.tolist()

# ── Teacher workload heuristic ────────────────────────────────────────────────
teacher_workload = mk_df_raw['teacher_id'].value_counts().to_dict()
mk_df_raw['teacher_degree'] = mk_df_raw['teacher_id'].map(teacher_workload)

# ── Pre-build global slot lists (computed ONCE) ───────────────────────────────
PRAC_SLOTS = [(r, s) for r in rooms_prac for s in valid_starts]
THEO_SLOTS = [(r, s) for r in rooms_theo for s in valid_starts]

# ── Pre-extracted numpy arrays for O(1) hot-loop access ──────────────────────
_REQ_PRAC   = mk_df_raw['requires_practicum_room'].to_numpy()
_TEACHER    = mk_df_raw['teacher_id'].to_numpy()
_T_PRIORITY = mk_df_raw['teacher_priority'].to_numpy()
_T_DEGREE   = mk_df_raw['teacher_degree'].to_numpy()
_TS_DAY     = timeslots_df['day'].to_numpy()
_TS_SESSION = timeslots_df['session_type'].to_numpy()
_N_COURSES  = len(mk_df_raw)

SESSION_MORNING   = (_TS_SESSION == 'Morning')
SESSION_AFTERNOON = (_TS_SESSION == 'Afternoon')

# ── HC weight: any 1 HC violation > any possible SC total ────────────────────
HC_WEIGHT = 10_000_000


# ==========================================
# 2. Fitness Function with Conflict-Directed Repair
# ==========================================
def fitness_function(sol_x, verbose_repair=False):
    """
    Three-phase decoder.

    PHASE 1 — Greedy assignment (Most-Constrained-First + PO random key).
    PHASE 2 — Conflict-Directed Repair: courses that Phase 1 couldn't place are
              rescued by temporarily relocating the assignment that blocks them.
    PHASE 3 — Full SC evaluation.
    """
    sc_value        = 0
    sc_count        = 0
    assignments     = []
    unassigned_idxs = []

    room_time_booked    = set()
    teacher_time_booked = set()
    teacher_day_rooms   = {}
    teacher_ts_to_asgn  = {}   # (teacher, ts_idx) → index in assignments

    avail_prac = set(range(len(PRAC_SLOTS)))
    avail_theo = set(range(len(THEO_SLOTS)))

    repair_log = []   # populated only when verbose_repair=True

    # ── MCF + PO random-key ordering ─────────────────────────────────────────
    base_scores = (_REQ_PRAC.astype(float) * 1000) + (_T_DEGREE * 10)
    priorities  = base_scores + (sol_x * 100)
    queue_order = np.argsort(-priorities)

    # =========================================================================
    # PHASE 1 — GREEDY ASSIGNMENT
    # =========================================================================
    for idx in queue_order:
        req_prac   = bool(_REQ_PRAC[idx])
        teacher    = _TEACHER[idx]
        t_priority = int(_T_PRIORITY[idx])

        slot_list = PRAC_SLOTS if req_prac else THEO_SLOTS
        avail_set = avail_prac  if req_prac else avail_theo

        hc_fallback    = None
        best_sc        = float('inf')
        best_slot_info = None
        sc_scan_count  = 0

        for si in list(avail_set):
            r, s = slot_list[si]
            t1, t2 = s, s + 1
            if (teacher, t1) in teacher_time_booked or (teacher, t2) in teacher_time_booked:
                continue
            if hc_fallback is None:
                hc_fallback = (r, s, _TS_DAY[s], t1, t2, si)
            day     = _TS_DAY[s]
            temp_sc = 0
            if t_priority == 1 and not (SESSION_MORNING[t1] and SESSION_MORNING[t2]):
                temp_sc += 10
            elif t_priority == 2 and not (SESSION_AFTERNOON[t1] and SESSION_AFTERNOON[t2]):
                temp_sc += 10
            if teacher in teacher_day_rooms and day in teacher_day_rooms[teacher]:
                if r not in teacher_day_rooms[teacher][day]:
                    temp_sc += 50
            if temp_sc < best_sc:
                best_sc        = temp_sc
                best_slot_info = (r, s, day, t1, t2, si)
                if temp_sc == 0:
                    break
            sc_scan_count += 1
            if sc_scan_count >= 50:
                break

        chosen = best_slot_info if best_slot_info is not None else hc_fallback
        if chosen is not None:
            r, s, day, t1, t2, si = chosen
            _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
                       assignments, room_time_booked, teacher_time_booked,
                       teacher_day_rooms, teacher_ts_to_asgn, avail_prac, avail_theo)
        else:
            unassigned_idxs.append(int(idx))

    # =========================================================================
    # PHASE 2 — CONFLICT-DIRECTED REPAIR
    # =========================================================================
    still_unassigned = []

    for idx in unassigned_idxs:
        req_prac    = bool(_REQ_PRAC[idx])
        teacher     = _TEACHER[idx]
        t_priority  = int(_T_PRIORITY[idx])
        course_name = mk_df_raw['course_name'].iloc[idx] if verbose_repair else None

        slot_list = PRAC_SLOTS if req_prac else THEO_SLOTS
        placed              = False
        attempts            = 0
        failed_relocations  = 0
        trace_steps         = []

        for r, s in slot_list:
            t1, t2 = s, s + 1
            if (r, t1) in room_time_booked or (r, t2) in room_time_booked:
                continue

            blocked_t1 = (teacher, t1) in teacher_time_booked
            blocked_t2 = (teacher, t2) in teacher_time_booked

            # ── Case A: slot is actually free (Phase 1 missed it) ────────────
            if not blocked_t1 and not blocked_t2:
                day = _TS_DAY[s]
                _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
                           assignments, room_time_booked, teacher_time_booked,
                           teacher_day_rooms, teacher_ts_to_asgn, avail_prac, avail_theo)
                placed = True
                if verbose_repair:
                    trace_steps.append({'action': 'DIRECT',
                        'detail': (f"Assigned directly → Room {rooms_df.iloc[r]['room_id']} | "
                                   f"{_TS_DAY[s]} {timeslots_df.iloc[s]['time_start']}–"
                                   f"{timeslots_df.iloc[t2]['time_end']}")})
                break

            # ── Case B: teacher is blocked — try to relocate the blocker ─────
            blocker_key      = (teacher, t1) if blocked_t1 else (teacher, t2)
            blocker_asgn_idx = teacher_ts_to_asgn.get(blocker_key)
            if blocker_asgn_idx is None:
                continue

            blocker    = assignments[blocker_asgn_idx]
            b_idx      = blocker['course_i']
            b_req_prac = bool(_REQ_PRAC[b_idx])
            b_teacher  = blocker['teacher']
            b_t1_old   = blocker['start_slot']
            b_t2_old   = blocker['end_slot']
            b_r_old    = blocker['room']
            b_priority = blocker['priority']
            b_day_old  = blocker['day']
            b_cname    = mk_df_raw['course_name'].iloc[b_idx] if verbose_repair else ''
            attempts  += 1

            # Temporarily unbook blocker
            room_time_booked.discard((b_r_old, b_t1_old))
            room_time_booked.discard((b_r_old, b_t2_old))
            teacher_time_booked.discard((b_teacher, b_t1_old))
            teacher_time_booked.discard((b_teacher, b_t2_old))
            teacher_ts_to_asgn.pop((b_teacher, b_t1_old), None)
            teacher_ts_to_asgn.pop((b_teacher, b_t2_old), None)
            if b_teacher in teacher_day_rooms and b_day_old in teacher_day_rooms[b_teacher]:
                try:
                    teacher_day_rooms[b_teacher][b_day_old].remove(b_r_old)
                except ValueError:
                    pass
                if not teacher_day_rooms[b_teacher][b_day_old]:
                    del teacher_day_rooms[b_teacher][b_day_old]

            b_slot_list = PRAC_SLOTS if b_req_prac else THEO_SLOTS
            b_avail_set = avail_prac  if b_req_prac else avail_theo
            b_si = -1
            try:
                b_si = b_slot_list.index((b_r_old, b_t1_old))
                b_avail_set.add(b_si)
            except ValueError:
                pass

            # Find new slot for blocker
            reloc_found = None
            for si2 in list(avail_prac if b_req_prac else avail_theo):
                r2, s2   = b_slot_list[si2]
                t1n, t2n = s2, s2 + 1
                if (r2, t1n) in room_time_booked or (r2, t2n) in room_time_booked:
                    continue
                if (b_teacher, t1n) in teacher_time_booked or (b_teacher, t2n) in teacher_time_booked:
                    continue
                reloc_found = (r2, s2, _TS_DAY[s2], t1n, t2n, si2)
                break

            if reloc_found is not None:
                r2, s2, day2, t1n, t2n, si2 = reloc_found
                _do_assign(b_idx, r2, s2, day2, t1n, t2n, b_priority, b_teacher,
                           b_req_prac, assignments, room_time_booked, teacher_time_booked,
                           teacher_day_rooms, teacher_ts_to_asgn, avail_prac, avail_theo,
                           update_idx=blocker_asgn_idx)
                day = _TS_DAY[s]
                _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
                           assignments, room_time_booked, teacher_time_booked,
                           teacher_day_rooms, teacher_ts_to_asgn, avail_prac, avail_theo)
                placed = True
                if verbose_repair:
                    trace_steps.append({'action': 'SWAP_SUCCESS', 'detail': (
                        f"Blocker '{b_cname}' (T:{b_teacher}) relocated "
                        f"{rooms_df.iloc[b_r_old]['room_id']} {_TS_DAY[b_t1_old]} "
                        f"{timeslots_df.iloc[b_t1_old]['time_start']}–{timeslots_df.iloc[b_t2_old]['time_end']}"
                        f"  →  "
                        f"{rooms_df.iloc[r2]['room_id']} {day2} "
                        f"{timeslots_df.iloc[s2]['time_start']}–{timeslots_df.iloc[t2n]['time_end']}"
                    )})
                break
            else:
                failed_relocations += 1
                if verbose_repair:
                    trace_steps.append({'action': 'SWAP_FAILED', 'detail': (
                        f"Blocker '{b_cname}' (T:{b_teacher}) at "
                        f"{rooms_df.iloc[b_r_old]['room_id']} {_TS_DAY[b_t1_old]} "
                        f"{timeslots_df.iloc[b_t1_old]['time_start']} "
                        f"— no alternative slot available"
                    )})
                # Restore blocker
                room_time_booked.add((b_r_old, b_t1_old))
                room_time_booked.add((b_r_old, b_t2_old))
                teacher_time_booked.add((b_teacher, b_t1_old))
                teacher_time_booked.add((b_teacher, b_t2_old))
                teacher_ts_to_asgn[(b_teacher, b_t1_old)] = blocker_asgn_idx
                teacher_ts_to_asgn[(b_teacher, b_t2_old)] = blocker_asgn_idx
                if b_teacher not in teacher_day_rooms:
                    teacher_day_rooms[b_teacher] = {}
                if b_day_old not in teacher_day_rooms[b_teacher]:
                    teacher_day_rooms[b_teacher][b_day_old] = []
                teacher_day_rooms[b_teacher][b_day_old].append(b_r_old)
                if b_si >= 0:
                    b_avail_set.discard(b_si)

        if verbose_repair:
            repair_log.append({
                'course_i':     idx,
                'course_name':  course_name,
                'teacher':      teacher,
                'type':         'Practicum' if req_prac else 'Theory',
                'placed':       placed,
                'attempts':     attempts,
                'failed_relocs': failed_relocations,
                'steps':        trace_steps,
            })

        if not placed:
            still_unassigned.append(idx)

    hc_count = len(still_unassigned)

    # =========================================================================
    # PHASE 3 — SC EVALUATION
    # =========================================================================
    for asgn in assignments:
        t1 = asgn['start_slot']
        t2 = asgn['end_slot']
        tp = asgn['priority']
        if tp == 1 and not (SESSION_MORNING[t1] and SESSION_MORNING[t2]):
            sc_value += 10
        elif tp == 2 and not (SESSION_AFTERNOON[t1] and SESSION_AFTERNOON[t2]):
            sc_value += 10

    for teacher_id, days in teacher_day_rooms.items():
        for day, rooms in days.items():
            unique_rooms = len(set(rooms))
            if unique_rooms > 1:
                v = unique_rooms - 1
                sc_count += v
                sc_value += 50 * v

    fitness = hc_count * HC_WEIGHT + sc_value
    metrics = {
        'hc_count':   hc_count,
        'hc_value':   hc_count * HC_WEIGHT,
        'sc_count':   sc_count,
        'sc_value':   sc_value,
        'repaired':   len(unassigned_idxs) - hc_count,
        'unassigned': len(unassigned_idxs),
        'repair_log': repair_log,
    }
    return fitness, assignments, metrics


# ==========================================
# 3. Assignment Helper
# ==========================================
def _do_assign(idx, r, s, day, t1, t2, t_priority, teacher, req_prac,
               assignments, room_time_booked, teacher_time_booked,
               teacher_day_rooms, teacher_ts_to_asgn,
               avail_prac, avail_theo, update_idx=None):
    room_time_booked.add((r, t1)); room_time_booked.add((r, t2))
    teacher_time_booked.add((teacher, t1)); teacher_time_booked.add((teacher, t2))

    slot_list = PRAC_SLOTS if req_prac else THEO_SLOTS
    avail_set = avail_prac  if req_prac else avail_theo
    for si2 in list(avail_set):
        r2, s2 = slot_list[si2]
        if r2 == r and (s2 == t1 or s2 == t2 or s2 + 1 == t1):
            avail_set.discard(si2)

    if teacher not in teacher_day_rooms:
        teacher_day_rooms[teacher] = {}
    if day not in teacher_day_rooms[teacher]:
        teacher_day_rooms[teacher][day] = []
    teacher_day_rooms[teacher][day].append(r)

    asgn = {'course_i': int(idx), 'room': r, 'start_slot': s, 'end_slot': t2,
             'teacher': teacher, 'priority': t_priority, 'day': day}

    if update_idx is not None:
        assignments[update_idx] = asgn
        teacher_ts_to_asgn[(teacher, t1)] = update_idx
        teacher_ts_to_asgn[(teacher, t2)] = update_idx
    else:
        new_idx = len(assignments)
        assignments.append(asgn)
        teacher_ts_to_asgn[(teacher, t1)] = new_idx
        teacher_ts_to_asgn[(teacher, t2)] = new_idx


def _eval_worker(x):
    return fitness_function(x, verbose_repair=False)


# ==========================================
# 4. Solution Class
# ==========================================
class Solution:
    __slots__ = ('X', 'Cost', 'assignments', 'metrics')

    def __init__(self, x, cost, assignments=None, metrics=None):
        self.X           = x
        self.Cost        = cost
        self.assignments = assignments
        self.metrics     = metrics

    def clone(self):
        return Solution(self.X.copy(), self.Cost, self.assignments, self.metrics)


# ==========================================
# 5. Parallel Population Evaluator
# ==========================================
def parallel_eval(vectors, n_workers=None):
    n_workers = n_workers or min(os.cpu_count(), len(vectors))
    if n_workers <= 1 or len(vectors) <= 4:
        return [fitness_function(v) for v in vectors]
    results = [None] * len(vectors)
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_eval_worker, v): i for i, v in enumerate(vectors)}
        for fut in as_completed(futures):
            results[futures[fut]] = fut.result()
    return results


# ==========================================
# 6. Display Helpers
# ==========================================
SEP  = "=" * 95

def print_iter(Iter, Sol, Best, elapsed):
    """Single iteration log line — matches the style of the reference output."""
    worst = max(s.Cost for s in Sol)
    m     = Best.metrics
    print(
        f"Iter {Iter:03d} | "
        f"Best: {Best.Cost:>12,.0f} | Worst: {worst:>12,.0f} | "
        f"HC: {m['hc_count']:>3d} ({m['hc_value']:>12,.0f}) | "
        f"SC: {m['sc_count']:>3d} ({m['sc_value']:>7,.0f}) | "
        f"Time: {elapsed:.1f}s"
    )


def print_repair_report(metrics, label="FINAL SOLUTION"):
    """
    Detailed per-course repair report — printed once after the run ends.
    Shows every course that went through Phase 2, what happened to it,
    and the exact swap steps taken.
    """
    log        = metrics.get('repair_log', [])
    n_sent     = metrics['unassigned']
    n_repaired = metrics['repaired']
    n_failed   = metrics['hc_count']

    print()
    print(SEP)
    print(f"  PHASE 2 REPAIR REPORT — {label}".center(95))
    print(f"  Courses sent to repair : {n_sent}".center(95))
    print(f"  Successfully repaired  : {n_repaired}".center(95))
    print(f"  Still unassigned (HC)  : {n_failed}".center(95))
    print(SEP)

    if not log:
        print("  (No repair trace available — re-run with verbose_repair=True)")
        print(SEP)
        return

    placed_entries = [e for e in log if e['placed']]
    failed_entries = [e for e in log if not e['placed']]

    # ── Successfully repaired courses ────────────────────────────────────────
    if placed_entries:
        print(f"\n  {'─'*40}  REPAIRED ({len(placed_entries)})  {'─'*40}")
        for i, e in enumerate(placed_entries, 1):
            print(f"\n  [{i:02d}] PLACED  | {e['course_name']:<40} | Teacher: {e['teacher']} | {e['type']}")
            print(f"       Swap attempts: {e['attempts']}   Failed relocations: {e['failed_relocs']}")
            for step in e['steps']:
                icon = "  ✔" if step['action'] != 'SWAP_FAILED' else "  ✘"
                print(f"  {icon} [{step['action']:>14s}]  {step['detail']}")

    # ── Courses that could not be placed ─────────────────────────────────────
    if failed_entries:
        print(f"\n  {'─'*39}  UNRESOLVED ({len(failed_entries)})  {'─'*39}")
        for i, e in enumerate(failed_entries, 1):
            print(f"\n  [{i:02d}] FAILED  | {e['course_name']:<40} | Teacher: {e['teacher']} | {e['type']}")
            print(f"       Swap attempts: {e['attempts']}   Failed relocations: {e['failed_relocs']}")
            for step in e['steps']:
                icon = "  ✔" if step['action'] != 'SWAP_FAILED' else "  ✘"
                print(f"  {icon} [{step['action']:>14s}]  {step['detail']}")
            if not e['steps']:
                print("       (No HC-free room slot exists for this room type — capacity exceeded)")

    print()
    print(SEP)


# ==========================================
# 7. Puma Optimizer — MECHANISM UNCHANGED
# ==========================================
def exploration(Sol, lb, ub, dim, nSol, n_workers):
    pCR = 0.20
    PCR = 1 - pCR
    p   = PCR / nSol
    Sol.sort(key=lambda s: s.Cost)
    new_vectors = []
    for i in range(nSol):
        x = Sol[i].X
        A = list(range(nSol)); A.remove(i); np.random.shuffle(A)
        a, b, c, d, e, f = A[:6]
        G = 2 * np.random.rand() - 1
        if np.random.rand() < 0.5:
            y = np.random.rand(dim) * (ub - lb) + lb
        else:
            da = Sol[a].X - Sol[b].X
            db = Sol[c].X - Sol[d].X
            dc = Sol[e].X - Sol[f].X
            y  = Sol[a].X + G * da + G * ((da - db) + (db - dc))
        y  = np.clip(y, lb, ub)
        z  = x.copy()
        j0 = np.random.randint(dim)
        mask = (np.random.rand(dim) <= pCR)
        mask[j0] = True
        z[mask] = y[mask]
        new_vectors.append(z)
    results = parallel_eval(new_vectors, n_workers)
    for i, (cost, asg, met) in enumerate(results):
        candidate = Solution(new_vectors[i], cost, asg, met)
        if candidate.Cost < Sol[i].Cost:
            Sol[i] = candidate
        else:
            pCR += p
    return Sol


def exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers):
    Q, Beta = 0.67, 2
    mbest   = np.mean([s.X for s in Sol], axis=0)
    new_vectors = []
    for i in range(nSol):
        beta1 = 2 * np.random.rand()
        beta2 = np.random.randn(dim)
        w, v  = np.random.randn(dim), np.random.randn(dim)
        F1    = np.random.randn(dim) * np.exp(2 - Iter * (2 / MaxIter))
        F2    = w * (v ** 2) * np.cos(2 * np.random.rand() * w)
        R_1   = 2 * np.random.rand() - 1
        S1    = (2 * np.random.rand() - 1 + np.random.randn(dim))
        S2    = F1 * R_1 * Sol[i].X + F2 * (1 - R_1) * Best.X
        VEC   = S2 / (S1 + 1e-10)
        if np.random.rand() <= 0.5:
            if np.random.rand() > Q:
                r_idx = np.random.randint(nSol)
                x_new = Best.X + beta1 * np.exp(beta2) * (Sol[r_idx].X - Sol[i].X)
            else:
                x_new = beta1 * VEC - Best.X
        else:
            r1    = np.random.randint(nSol)
            sign  = (-1) ** np.random.randint(0, 2)
            x_new = (mbest * Sol[r1].X - sign * Sol[i].X) / (1 + Beta * np.random.rand())
        new_vectors.append(np.clip(x_new, lb, ub))
    results = parallel_eval(new_vectors, n_workers)
    for i, (cost, asg, met) in enumerate(results):
        candidate = Solution(new_vectors[i], cost, asg, met)
        if candidate.Cost < Sol[i].Cost:
            Sol[i] = candidate
    return Sol


def run_puma_optimizer(nSol, MaxIter, lb, ub, dim, n_workers=None):
    n_workers = n_workers or min(os.cpu_count(), nSol)
    print(f"[INFO] Parallel workers: {n_workers}")

    # ── Initial population ────────────────────────────────────────────────────
    print("Mengevaluasi populasi inisial...")
    init_vectors    = [lb + (ub - lb) * np.random.rand(dim) for _ in range(nSol)]
    init_vectors[0] = np.full(dim, 0.5)    # seed: pure MCF order

    results      = parallel_eval(init_vectors, n_workers)
    Sol          = [Solution(init_vectors[i], *results[i]) for i in range(nSol)]
    Best         = min(Sol, key=lambda s: s.Cost)
    Initial_Best = Best.clone()

    m = Best.metrics
    print(f"\nInitial Best Fitness: {Best.Cost:,.0f} | "
          f"HC: {m['hc_count']} ({m['hc_value']:,}) | "
          f"SC: {m['sc_count']} ({m['sc_value']:,})")

    if Best.metrics['hc_count'] == 0:
        print("[INFO] Zero HC violations in initial population — skipping optimisation.")
        return Best

    # ── PO state variables (UNCHANGED) ───────────────────────────────────────
    UnSelected = [1, 1]
    F3_Explore = F3_Exploit = 0
    Seq_Time_Explore  = [1, 1, 1];  Seq_Time_Exploit  = [1, 1, 1]
    Seq_Cost_Explore  = [1, 1, 1];  Seq_Cost_Exploit  = [1, 1, 1]
    PF            = [0.5, 0.5, 0.3]
    PF_F3         = []
    Mega_Explor   = Mega_Exploit = 0.99
    Costs_Explor  = np.zeros(3)
    Costs_Exploit = np.zeros(3)
    Flag_Change   = 1

    print()
    print(SEP)
    print("MEMULAI PROSES OPTIMASI PUMA (v6)".center(95))
    print(SEP)

    t_start    = time.perf_counter()
    stagnation = 0
    prev_best  = Best.Cost

    # ── Bootstrap iterations 1–3 ─────────────────────────────────────────────
    for Iter in range(1, 4):
        Sol_E1 = exploration ([s.clone() for s in Sol], lb, ub, dim, nSol, n_workers)
        Sol_E2 = exploitation([s.clone() for s in Sol], lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers)
        Costs_Explor[Iter - 1]  = min(s.Cost for s in Sol_E1)
        Costs_Exploit[Iter - 1] = min(s.Cost for s in Sol_E2)
        merged = Sol + Sol_E1 + Sol_E2
        merged.sort(key=lambda s: s.Cost)
        Sol  = merged[:nSol]
        Best = Sol[0]
        print_iter(Iter, Sol, Best, time.perf_counter() - t_start)
        if Best.metrics['hc_count'] == 0:
            print(f"\n[INFO] Zero HC achieved at bootstrap iteration {Iter} — stopping early.")
            return Best

    Seq_Cost_Explore[0] = abs(Initial_Best.Cost  - Costs_Explor[0])
    Seq_Cost_Explore[1] = abs(Costs_Explor[1]    - Costs_Explor[0])
    Seq_Cost_Explore[2] = abs(Costs_Explor[2]    - Costs_Explor[1])
    Seq_Cost_Exploit[0] = abs(Initial_Best.Cost  - Costs_Exploit[0])
    Seq_Cost_Exploit[1] = abs(Costs_Exploit[1]   - Costs_Exploit[0])
    Seq_Cost_Exploit[2] = abs(Costs_Exploit[2]   - Costs_Exploit[1])
    for v in Seq_Cost_Explore + Seq_Cost_Exploit:
        if v != 0: PF_F3.append(v)

    # ── Main loop (PO mechanism UNCHANGED) ───────────────────────────────────
    for Iter in range(4, MaxIter + 1):
        F1_Explor  = PF[0] * (Seq_Cost_Explore[0] / Seq_Time_Explore[0])
        F1_Exploit = PF[0] * (Seq_Cost_Exploit[0] / Seq_Time_Exploit[0])
        F2_Explor  = PF[1] * (sum(Seq_Cost_Explore) / sum(Seq_Time_Explore))
        F2_Exploit = PF[1] * (sum(Seq_Cost_Exploit) / sum(Seq_Time_Exploit))
        Score_Explore  = Mega_Explor  * (F1_Explor  + F2_Explor)
        Score_Exploit  = Mega_Exploit * (F1_Exploit + F2_Exploit)

        if Score_Explore > Score_Exploit:
            SelectFlag   = 1
            Sol          = exploration(Sol, lb, ub, dim, nSol, n_workers)
            Count_select = UnSelected.copy()
            UnSelected[1] += 1;  UnSelected[0] = 1
            F3_Explore    = PF[2];  F3_Exploit += PF[2]
            TBest         = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Explore[2], Seq_Cost_Explore[1] = Seq_Cost_Explore[1], Seq_Cost_Explore[0]
            Seq_Cost_Explore[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Explore[0]: PF_F3.append(Seq_Cost_Explore[0])
        else:
            SelectFlag   = 2
            Sol          = exploitation(Sol, lb, ub, dim, nSol, Best, MaxIter, Iter, n_workers)
            Count_select = UnSelected.copy()
            UnSelected[0] += 1;  UnSelected[1] = 1
            F3_Explore   += PF[2];  F3_Exploit = PF[2]
            TBest         = min(Sol, key=lambda s: s.Cost)
            Seq_Cost_Exploit[2], Seq_Cost_Exploit[1] = Seq_Cost_Exploit[1], Seq_Cost_Exploit[0]
            Seq_Cost_Exploit[0] = abs(Best.Cost - TBest.Cost)
            if Seq_Cost_Exploit[0]: PF_F3.append(Seq_Cost_Exploit[0])

        if TBest.Cost < Best.Cost:
            Best = TBest

        if Flag_Change != SelectFlag:
            Flag_Change = SelectFlag
            Seq_Time_Explore[2], Seq_Time_Explore[1], Seq_Time_Explore[0] = \
                Seq_Time_Explore[1], Seq_Time_Explore[0], Count_select[0]
            Seq_Time_Exploit[2], Seq_Time_Exploit[1], Seq_Time_Exploit[0] = \
                Seq_Time_Exploit[1], Seq_Time_Exploit[0], Count_select[1]

        if Score_Explore < Score_Exploit:
            Mega_Explor  = max(Mega_Explor  - 0.01, 0.01); Mega_Exploit = 0.99
        else:
            Mega_Explor  = 0.99; Mega_Exploit = max(Mega_Exploit - 0.01, 0.01)

        print_iter(Iter, Sol, Best, time.perf_counter() - t_start)

        if Best.metrics['hc_count'] == 0:
            print(f"\n[INFO] Zero HC violations achieved at iteration {Iter} — stopping early.")
            break

        # ── Stagnation recovery ───────────────────────────────────────────────
        if Best.Cost >= prev_best:
            stagnation += 1
        else:
            stagnation = 0
            prev_best  = Best.Cost

        if stagnation >= 10:
            print(f"[WARN] Stagnation detected at iter {Iter} — injecting diversity...")
            n_replace = nSol // 3
            new_vecs  = [lb + (ub - lb) * np.random.rand(dim) for _ in range(n_replace)]
            res       = parallel_eval(new_vecs, n_workers)
            for k, (c, a, met) in enumerate(res):
                Sol[-(k + 1)] = Solution(new_vecs[k], c, a, met)
            stagnation = 0

    total_time = time.perf_counter() - t_start
    print(f"\n[INFO] Optimasi selesai dalam {total_time:.1f}s")
    return Best


# ==========================================
# 8. Eksekusi & Ekspor
# ==========================================
if __name__ == '__main__':
    nSol    = 100
    MaxIter = 100
    dim     = _N_COURSES
    lb, ub  = 0.0, 1.0

    t0            = time.perf_counter()
    best_solution = run_puma_optimizer(nSol, MaxIter, lb, ub, dim, n_workers=None)
    total_elapsed = time.perf_counter() - t0
    print(f"\nTotal elapsed: {total_elapsed:.1f}s")

    # ── Repair report (re-evaluate best solution with verbose trace) ──────────
    if best_solution.metrics['unassigned'] > 0:
        print("\n[INFO] Re-evaluating best solution for repair trace...")
        _, _, m_verbose = fitness_function(best_solution.X, verbose_repair=True)
        print_repair_report(m_verbose, label="FINAL BEST SOLUTION")
    else:
        print("\n[INFO] All courses assigned — no repair report needed.")

    # ── Build result DataFrame ────────────────────────────────────────────────
    res_list = []
    for a in best_solution.assignments:
        course = mk_df_raw.iloc[a['course_i']]
        room   = rooms_df.iloc[a['room']]
        t1     = timeslots_df.iloc[a['start_slot']]
        t2     = timeslots_df.iloc[a['end_slot']]
        res_list.append({
            'event_id':    course['event_id'],
            'course_name': course['course_name'],
            'teacher_id':  course['teacher_id'],
            'room_id':     room['room_id'],
            'day':         t1['day'],
            'time_start':  t1['time_start'],
            'time_end':    t2['time_end'],
        })

    res_df   = pd.DataFrame(res_list)
    out_path = 'scheduled_courses_poa_v6.csv'
    res_df.to_csv(out_path, index=False)

    m = best_solution.metrics
    print()
    print(SEP)
    print("  FINAL RESULT".center(95))
    print(SEP)
    print(f"  HC Violations  : {m['hc_count']}   (penalty: {m['hc_value']:,})")
    print(f"  SC Violations  : {m['sc_count']}   (penalty: {m['sc_value']:,})")
    print(f"  Total Fitness  : {best_solution.Cost:,}")
    print(f"  Courses scheduled : {len(res_list)} / {_N_COURSES}")
    print(f"  Repaired by P2    : {m['repaired']} / {m['unassigned']}")
    print(SEP)
    print(f"[INFO] Solusi disimpan di '{out_path}'")

[INFO] Memuat dan memproses dataset...
[INFO] Parallel workers: 8
Mengevaluasi populasi inisial...

Initial Best Fitness: 5,370 | HC: 0 (0) | SC: 55 (5,370)
[INFO] Zero HC violations in initial population — skipping optimisation.

Total elapsed: 3.7s

[INFO] Re-evaluating best solution for repair trace...

                           PHASE 2 REPAIR REPORT — FINAL BEST SOLUTION                         
                                   Courses sent to repair : 40                                 
                                   Successfully repaired  : 40                                 
                                    Still unassigned (HC)  : 0                                 

  ────────────────────────────────────────  REPAIRED (40)  ────────────────────────────────────────

  [01] PLACED  | Probabilitas dan Statistika              | Teacher: 16 | Theory
       Swap attempts: 1   Failed relocations: 0
    ✔ [  SWAP_SUCCESS]  Blocker 'Pemrograman SQL - Praktikum' (T:16) relocate